[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/04_line_search_newton_quasi_newton/exercises.ipynb)

# Module 04 — Line Search, Newton and Quasi-Newton Methods — Exercises

Setup cell: the imports, the seeded generator and the print options used by every verification cell below.

In [1]:
import numpy as np
from scipy.optimize import line_search

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

## L0 — Concept Checks

### Problem L0.1 — Why Plain Decrease Is Not Enough

**Problem Statement:** True or false: if $f(\mathbf{x}_{k+1}) \lt f(\mathbf{x}_k)$ at every iteration and
$f$ is bounded below, then $\mathbf{x}_k$ converges to a stationary point. Give a counterexample and state
the repair.

*Intuition:* A sequence of decreases can be summable — the algorithm runs out of progress before it runs out of gradient.

**Solution:**

**False.** Consider $f(x) = x^2$ on $\mathbb{R}$ and the sequence

$$
x_k = 1 + 2^{-k}, \qquad f(x_k) = \left(1 + 2^{-k}\right)^2
$$

Every step decreases $f$ strictly, $f$ is bounded below by $0$, and yet
$x_k \to 1$ with $f'(1) = 2 \neq 0$: the limit is not stationary at all. The total decrease
$f(x_0) - \lim_k f(x_k) = 4 - 1 = 3$ is finite and is exhausted long before the gradient dies.

**Why it happens.** Decrease per step is $f(x_k) - f(x_{k+1}) = O(2^{-k})$, which is summable, while the
gradient stays bounded away from $0$. Nothing ties the *size* of the decrease to the *size* of the
gradient.

**The repair (Armijo sufficient decrease).** Require the decrease to be proportional to the step length
and the directional slope: with $c_1 \in (0,1)$,

$$
f(\mathbf{x}_k + \alpha_k\mathbf{d}_k) \le f(\mathbf{x}_k) + c_1\alpha_k\, \nabla f(\mathbf{x}_k)^T \mathbf{d}_k
$$

Summing over $k$ then bounds $\sum_k \alpha_k \lvert \nabla f(\mathbf{x}_k)^T\mathbf{d}_k\rvert$ by the total
decrease $f(\mathbf{x}_0) - f^*$, which forces $\alpha_k \nabla f_k^T\mathbf{d}_k \to 0$ — the seed of the
Zoutendijk condition (Problem L3.2).

$$
\boxed{\text{monotone decrease} \not\Rightarrow \text{stationarity}; \ \text{Armijo ties decrease to } \alpha_k \lvert \nabla f_k^T \mathbf{d}_k \rvert}
$$

> **Key takeaway:** Convergence proofs never use "the loss went down" — they use "the loss went down by *at least* a fixed fraction of the predicted linear decrease", which is exactly what the Armijo inequality asserts.

**Verification.** The cell below recomputes every number claimed in Problem L0.1.

In [2]:
# f(x) = x^2 with x_k = 1 + 2^-k: strict decrease, bounded below, non-stationary limit.
k = np.arange(0, 21)
xk = 1.0 + 2.0**(-k.astype(float))
fk = xk**2
print("f strictly decreasing at every step:", bool(np.all(np.diff(fk) < 0)))
print(f"f is bounded below by 0: min f_k = {fk.min():.6f}")
print(f"limit x_k = {xk[-1]:.10f},  f'(1) = {2*1.0}  (not stationary)")
print(f"total decrease f(x_0) - lim f(x_k) = {fk[0] - 1.0:.6f}")
print("per-step decreases (first 5):", np.array2string(-np.diff(fk)[:5], precision=6))
assert np.all(np.diff(fk) < 0) and abs(xk[-1] - 1.0) < 1e-6
assert abs((fk[0] - 1.0) - 3.0) < 1e-6

f strictly decreasing at every step: True
f is bounded below by 0: min f_k = 1.000002
limit x_k = 1.0000009537,  f'(1) = 2.0  (not stationary)
total decrease f(x_0) - lim f(x_k) = 3.000000
per-step decreases (first 5): [1.75     0.6875   0.296875 0.136719 0.06543 ]


### Problem L0.2 — What Each Wolfe Condition Actually Forbids

**Problem Statement:** State the (weak) Wolfe conditions with parameters $0 \lt c_1 \lt c_2 \lt 1$.
Explain precisely which failure mode each one rules out, why $c_1 \lt c_2$ is required, and what the
strong Wolfe variant adds.

*Intuition:* One inequality is an upper barrier on the function value, the other a lower barrier on the remaining slope.

**Solution:**

Let $\phi(\alpha) = f(\mathbf{x}_k + \alpha\mathbf{d}_k)$, so $\phi'(0) = \nabla f(\mathbf{x}_k)^T\mathbf{d}_k \lt 0$
for a descent direction.

**(W1) Armijo / sufficient decrease.**

$$
\phi(\alpha) \le \phi(0) + c_1\alpha\,\phi'(0)
$$

This forbids **overshooting**: $\alpha$ must stay under the line of slope $c_1\phi'(0)$, so steps that jump
past the valley and back up the far wall are rejected. Alone, it is satisfied by all sufficiently small
$\alpha$ — including uselessly small ones.

**(W2) Curvature condition.**

$$
\phi'(\alpha) \ge c_2\,\phi'(0)
$$

This forbids **undershooting**: the slope at the accepted point must have flattened to at most $c_2$ of its
initial steepness. If $\phi'(\alpha)$ is still very negative, the step is too short — there is obvious
remaining progress — and it is rejected.

**Why $c_1 \lt c_2$.** Steps satisfying both must exist. By the mean value theorem there is
$\alpha^\dagger$ with $\phi(\alpha) - \phi(0) = \alpha\,\phi'(\alpha^\dagger)$; matching this against
(W1) at the boundary requires the slope threshold $c_2\phi'(0)$ to sit *above* $c_1\phi'(0)$ (recall
$\phi'(0) \lt 0$, so larger $c$ means a less negative threshold). If $c_2 \le c_1$ the two constraints can
be incompatible, and the classical existence theorem (Problem L1.3) explicitly assumes $c_1 \lt c_2$.

**Strong Wolfe.** Replace (W2) by
$\lvert \phi'(\alpha)\rvert \le c_2\lvert \phi'(0)\rvert$, which additionally forbids the slope from
becoming *strongly positive* — i.e. it keeps $\alpha$ near a local minimizer of $\phi$ rather than far up
the opposite wall. Typical values: $c_1 = 10^{-4}$ (very permissive), $c_2 = 0.9$ for quasi-Newton,
$c_2 = 0.1$ for nonlinear conjugate gradient.

$$
\boxed{\text{(W1) blocks too-long steps; (W2) blocks too-short steps; both are needed, and } c_1 \lt c_2}
$$

> **Key takeaway:** The Wolfe pair brackets the step from both sides, which is what makes an *inexact* line search as safe as an exact one for convergence theory — at a cost of a couple of function evaluations instead of dozens.

**Verification.** The cell below recomputes every number claimed in Problem L0.2.

In [3]:
# phi(a) = f(x0 + a d0) for f = x^2 + 4y^2 at (1,1) along -grad f:  260 a^2 - 68 a + 5.
phi  = lambda a: 260.0*a**2 - 68.0*a + 5.0
dphi = lambda a: 520.0*a - 68.0
c1, c2 = 0.25, 0.9
W1 = lambda a: phi(a) <= phi(0.0) + c1*a*dphi(0.0)
W2 = lambda a: dphi(a) >= c2*dphi(0.0)

a_hi = (1 - c1)*68/260      # W1 holds for a <= a_hi
a_lo = (1 - c2)*68/520      # W2 holds for a >= a_lo
print(f"W1 holds on (0, {a_hi:.6f}];  W2 holds on [{a_lo:.6f}, inf)")
for a in [0.005, 0.05, 0.25]:
    print(f"  alpha={a:<6}: W1={str(W1(a)):<5} W2={str(W2(a)):<5}")
assert W1(0.005) and not W2(0.005)      # too short: passes decrease, fails curvature
assert W1(0.05) and W2(0.05)            # acceptable
assert not W1(0.25) and W2(0.25)        # too long: fails decrease

W1 holds on (0, 0.196154];  W2 holds on [0.013077, inf)
  alpha=0.005 : W1=True  W2=False
  alpha=0.05  : W1=True  W2=True 
  alpha=0.25  : W1=False W2=True 


### Problem L0.3 — When Is the Newton Direction a Descent Direction?

**Problem Statement:** For $\mathbf{p}_k = -\left[\nabla^2 f(\mathbf{x}_k)\right]^{-1}\nabla f(\mathbf{x}_k)$,
determine exactly when $\nabla f(\mathbf{x}_k)^T\mathbf{p}_k \lt 0$. Give an explicit example where the
Newton direction points uphill, and name two standard fixes.

*Intuition:* Newton trusts the local quadratic model — if the model is a saddle or an inverted bowl, its "minimizer" is nonsense.

**Solution:**

**Step 1 (the slope along the Newton direction).** Write $H = \nabla^2 f(\mathbf{x}_k)$ (invertible) and
$\mathbf{g} = \nabla f(\mathbf{x}_k) \neq \mathbf{0}$. Then

$$
\nabla f(\mathbf{x}_k)^T\mathbf{p}_k = -\mathbf{g}^T H^{-1}\mathbf{g}
$$

So the Newton direction is a descent direction **iff** $\mathbf{g}^T H^{-1}\mathbf{g} \gt 0$, which is
guaranteed for every $\mathbf{g}$ precisely when $H^{-1} \succ 0$, equivalently $H \succ 0$ (a symmetric
matrix and its inverse share the sign of the spectrum).

**Step 2 (an uphill example).** Take $f(x,y) = \frac{1}{2}x^2 - \frac{1}{2}y^2 + y$, so
$H = \operatorname{diag}(1, -1)$ (indefinite) and at $\mathbf{x}_k = (1, 0)$,
$\mathbf{g} = (1, 1)$. Then

$$
\mathbf{p}_k = -H^{-1}\mathbf{g} = -\begin{pmatrix} 1 \\ -1\end{pmatrix} = \begin{pmatrix} -1 \\ 1\end{pmatrix}, \qquad \mathbf{g}^T\mathbf{p}_k = -1 + 1 = 0
$$

and shifting to $\mathbf{g} = (1, 2)$ (take $f = \frac12 x^2 - \frac12 y^2 + 2y$ at $(1,0)$) gives
$\mathbf{p}_k = (-1, 2)$ and $\mathbf{g}^T\mathbf{p}_k = -1 + 4 = 3 \gt 0$: strictly **uphill**. In general
Newton moves *toward* the stationary point of the model, and when that stationary point is a saddle or a
maximum, "toward" can mean up.

**Step 3 (fixes).**

1. **Modified / damped Newton:** solve $(H + \tau I)\mathbf{p} = -\mathbf{g}$ with $\tau \gt 0$ large
   enough that $H + \tau I \succ 0$ (found by a trial Cholesky factorization); as $\tau \to \infty$ the
   direction rotates continuously into $-\mathbf{g}$.
2. **Quasi-Newton (BFGS):** build $B_k \succ 0$ by construction from the secant pairs, so the direction is
   *always* descent (Problem L3.3). Trust-region methods are a third fix: they bound $\lVert \mathbf{p}\rVert$
   so that even an indefinite model yields a usable step.

$$
\boxed{\nabla f^T\mathbf{p}_{\text{Newton}} = -\mathbf{g}^T H^{-1}\mathbf{g} \lt 0 \ \text{for all } \mathbf{g} \iff H \succ 0}
$$

> **Key takeaway:** Newton solves $\nabla f = \mathbf{0}$, not "minimize $f$" — the two coincide only where the Hessian is positive definite, which is why every practical Newton code carries a positive-definiteness safeguard.

**Verification.** The cell below recomputes every number claimed in Problem L0.3.

In [4]:
# f(x,y) = x^2/2 - y^2/2 + 2y at (1,0): indefinite Hessian, Newton direction points uphill.
H_ind = np.diag([1.0, -1.0])
g_ind = np.array([1.0, 2.0])            # grad = (x, -y + 2) at (1,0)
p_ind = -np.linalg.solve(H_ind, g_ind)
print("Hessian eigenvalues:", np.linalg.eigvalsh(H_ind))
print(f"Newton direction p = {p_ind},  grad.p = {g_ind @ p_ind:+.4f}  -> ASCENT")

tau = 1.5                                # modified Newton: H + tau I  is now SPD
p_mod = -np.linalg.solve(H_ind + tau*np.eye(2), g_ind)
print(f"modified (tau={tau}) p = {p_mod},  grad.p = {g_ind @ p_mod:+.4f}  -> descent")
assert g_ind @ p_ind > 0 and g_ind @ p_mod < 0
assert np.min(np.linalg.eigvalsh(H_ind)) < 0

Hessian eigenvalues: [-1.  1.]
Newton direction p = [-1.  2.],  grad.p = +3.0000  -> ASCENT
modified (tau=1.5) p = [-0.4 -4. ],  grad.p = -8.4000  -> descent


### Problem L0.4 — Linear, Superlinear, Quadratic — What the Words Cost

**Problem Statement:** Define linear, superlinear and quadratic convergence for
$e_k = \lVert \mathbf{x}_k - \mathbf{x}^*\rVert$. Starting from $e_0 = 10^{-1}$ with the constants
$e_{k+1} = 0.9\,e_k$ (linear), $e_{k+1} = e_k^{1.5}$ (superlinear) and $e_{k+1} = e_k^2$ (quadratic),
tabulate how many iterations each needs to reach $10^{-12}$.

*Intuition:* Linear buys a fixed number of digits per step; quadratic doubles the digit count per step.

**Solution:**

**Definitions.** With $e_k \to 0$ and $e_k \gt 0$:

- **Linear (rate $\rho \in (0,1)$):** $e_{k+1} \le \rho\, e_k$.
- **Superlinear:** $e_{k+1}/e_k \to 0$; the order-$q$ version is $e_{k+1} \le C e_k^{q}$ with $q \gt 1$.
- **Quadratic:** $e_{k+1} \le C e_k^2$.

Writing $d_k = -\log_{10} e_k$ for the number of correct digits, the three recursions read
$d_{k+1} = d_k + \log_{10}(1/\rho)$, $d_{k+1} = 1.5\,d_k$, and $d_{k+1} = 2 d_k$.

**Step 1 (linear, $\rho = 0.9$).** $d_0 = 1$ and each step adds
$\log_{10}(1/0.9) \approx 0.0458$ digits, so reaching $d = 12$ needs

$$
k \ge \frac{12 - 1}{0.0458} \approx 241 \ \text{iterations}
$$

**Step 2 (superlinear, $q = 1.5$).** $d_k = 1.5^k d_0 = 1.5^k$. Then
$d_1 = 1.5$, $d_2 = 2.25$, $d_3 = 3.375$, $d_4 = 5.06$, $d_5 = 7.59$, $d_6 = 11.39$, $d_7 = 17.09$:
**7 iterations**.

**Step 3 (quadratic).** $d_k = 2^k$: $1, 2, 4, 8, 16$ — **4 iterations**.

| Method class | Recursion on digits | Iterations to $10^{-12}$ |
|---|---|---|
| Gradient descent, ill-conditioned | $d_{k+1} = d_k + 0.046$ | $\approx 241$ |
| BFGS / L-BFGS (superlinear) | $d_{k+1} = 1.5\,d_k$ | $7$ |
| Newton (quadratic) | $d_{k+1} = 2 d_k$ | $4$ |

$$
\boxed{\text{linear} \approx 241, \quad \text{superlinear } (q = 1.5) = 7, \quad \text{quadratic} = 4 \ \text{iterations}}
$$

> **Key takeaway:** Quadratic convergence is not "twice as fast" — it is a different *kind* of arithmetic on the digit count, which is why the last few Newton iterations are essentially free and why the whole game is getting into the fast local region.

**Verification.** The cell below recomputes every number claimed in Problem L0.4.

In [5]:
# iterations to drive e_0 = 1e-1 down to 1e-12 under three error recursions
def steps_to(update, e0=1e-1, tol=1e-12, cap=10_000):
    e, k = e0, 0
    while e > tol and k < cap:
        e = update(e); k += 1
    return k

n_lin  = steps_to(lambda e: 0.9*e)
n_sup  = steps_to(lambda e: e**1.5)
n_quad = steps_to(lambda e: e**2)
print(f"linear    e_(k+1)=0.9 e_k    : {n_lin} iterations")
print(f"superlin. e_(k+1)=e_k^1.5    : {n_sup} iterations")
print(f"quadratic e_(k+1)=e_k^2      : {n_quad} iterations")
print(f"closed form for the linear case: ceil(11/log10(1/0.9)) = "
      f"{int(np.ceil(11/np.log10(1/0.9)))}")
assert (n_lin, n_sup, n_quad) == (241, 7, 4)

linear    e_(k+1)=0.9 e_k    : 241 iterations
superlin. e_(k+1)=e_k^1.5    : 7 iterations
quadratic e_(k+1)=e_k^2      : 4 iterations
closed form for the linear case: ceil(11/log10(1/0.9)) = 241


## L1 — Foundations

### Problem L1.1 — Backtracking by Hand on an Ill-Conditioned Quadratic

**Problem Statement:** For $f(x,y) = x^2 + 10y^2$ at $\mathbf{x}_0 = (1,1)$ with the steepest-descent
direction $\mathbf{d}_0 = -\nabla f(\mathbf{x}_0)$, run backtracking with $\bar{\alpha} = 1$,
$\rho = 0.5$, $c_1 = 10^{-4}$. Report every trial, the accepted step, the new iterate, and compare the
accepted step against the theoretical lower bound $\alpha \ge 2(1-c_1)\rho/L$.

*Intuition:* Backtracking discovers the curvature scale by trial and error — each halving is one bisection of the stability interval.

**Solution:**

**Step 0 (data).** $\nabla f = (2x, 20y)$, so $\nabla f(1,1) = (2, 20)$ and $\mathbf{d}_0 = (-2,-20)$.

$$
f(\mathbf{x}_0) = 11, \qquad \phi'(0) = \nabla f_0^T\mathbf{d}_0 = -(4 + 400) = -404
$$

The Armijo test is $\phi(\alpha) \le 11 - 10^{-4}\cdot 404\,\alpha = 11 - 0.0404\,\alpha$, where
$\phi(\alpha) = (1-2\alpha)^2 + 10(1-20\alpha)^2$.

**Step 1 (the trials).**

| $\alpha$ | $\phi(\alpha)$ | Armijo bound $11 - 0.0404\alpha$ | verdict |
|---|---|---|---|
| $1$ | $1 + 10(361) = 3611$ | $10.9596$ | reject |
| $0.5$ | $0 + 10(81) = 810$ | $10.9798$ | reject |
| $0.25$ | $0.25 + 10(16) = 160.25$ | $10.9899$ | reject |
| $0.125$ | $0.5625 + 10(2.25) = 23.0625$ | $10.9950$ | reject |
| $0.0625$ | $0.765625 + 10(0.0625) = 1.390625$ | $10.9975$ | **accept** |

**Step 2 (the new iterate).** With $\alpha_0 = 1/16$,

$$
\mathbf{x}_1 = (1,1) + \tfrac{1}{16}(-2,-20) = (0.875,\ -0.25), \qquad f(\mathbf{x}_1) = 1.390625
$$

a decrease from $11$ in five function evaluations.

**Step 3 (against the theoretical bound).** Here $\nabla^2 f = \operatorname{diag}(2,20)$, so $L = 20$. The
backtracking guarantee (Problem L1.2) is

$$
\alpha_{\text{accepted}} \ge \frac{2(1-c_1)\rho}{L} = \frac{2(0.9999)(0.5)}{20} = 0.049995
$$

and indeed $0.0625 \ge 0.05$. Note also the stability ceiling $2/L = 0.1$: backtracking landed inside
$(0, 2/L)$ automatically, without ever being told $L$.

$$
\boxed{\alpha_0 = \tfrac{1}{16},\ \mathbf{x}_1 = (0.875, -0.25),\ f = 1.3906; \ \text{5 evaluations, and } \alpha_0 \ge \tfrac{2(1-c_1)\rho}{L} = 0.05}
$$

> **Key takeaway:** Backtracking is an $L$-estimator in disguise: the number of halvings measures $\log_2(\bar{\alpha}L/2)$, so a run that always backtracks five times is telling you the curvature scale of your problem.

**Verification.** The cell below recomputes every number claimed in Problem L1.1.

In [6]:
# backtracking on f(x,y) = x^2 + 10 y^2 from (1,1), alpha_bar=1, rho=0.5, c1=1e-4
f11 = lambda v: v[0]**2 + 10.0*v[1]**2
g11 = lambda v: np.array([2.0*v[0], 20.0*v[1]])
x0 = np.array([1.0, 1.0]); d0 = -g11(x0)
c1, rho = 1e-4, 0.5
slope = g11(x0) @ d0
print(f"f(x0) = {f11(x0)},  phi'(0) = {slope}")

a, trials = 1.0, []
while f11(x0 + a*d0) > f11(x0) + c1*a*slope:
    trials.append((a, f11(x0 + a*d0), f11(x0) + c1*a*slope, "reject")); a *= rho
trials.append((a, f11(x0 + a*d0), f11(x0) + c1*a*slope, "ACCEPT"))
for t in trials:
    print(f"  alpha={t[0]:<8.5f} phi={t[1]:<12.6f} bound={t[2]:<10.6f} {t[3]}")

x1 = x0 + a*d0
L = 20.0
print(f"accepted alpha = {a} = 1/{round(1/a)},  x1 = {x1},  f(x1) = {f11(x1)}")
print(f"theoretical floor 2(1-c1)rho/L = {2*(1-c1)*rho/L:.6f};  stability ceiling 2/L = {2/L}")
assert a == 1/16 and np.allclose(x1, [0.875, -0.25]) and np.isclose(f11(x1), 1.390625)
assert len(trials) == 5 and a >= 2*(1-c1)*rho/L and a < 2/L

f(x0) = 11.0,  phi'(0) = -404.0
  alpha=1.00000  phi=3611.000000  bound=10.959600  reject
  alpha=0.50000  phi=810.000000   bound=10.979800  reject
  alpha=0.25000  phi=160.250000   bound=10.989900  reject
  alpha=0.12500  phi=23.062500    bound=10.994950  reject
  alpha=0.06250  phi=1.390625     bound=10.997475  ACCEPT
accepted alpha = 0.0625 = 1/16,  x1 = [ 0.875 -0.25 ],  f(x1) = 1.390625
theoretical floor 2(1-c1)rho/L = 0.049995;  stability ceiling 2/L = 0.1


### Problem L1.2 — Backtracking Terminates, With a Quantitative Floor

**Problem Statement:** Let $f$ be $L$-smooth and $\mathbf{d}_k$ a descent direction. Prove that the
backtracking loop $\alpha \leftarrow \rho\alpha$ terminates after finitely many halvings, and that the
accepted step satisfies $\alpha_k \ge \rho\,\dfrac{2(1-c_1)\lvert \nabla f_k^T\mathbf{d}_k\rvert}{L\lVert \mathbf{d}_k\rVert^2}$.
For the special case of steepest descent, $\mathbf{d}_k = -\nabla f_k$, this floor simplifies to
$\alpha_k \ge \min\left\{\bar{\alpha},\ \frac{2(1-c_1)\rho}{L}\right\}$.

*Intuition:* The descent lemma says the quadratic model is an upper bound, and a quadratic upper bound is beaten by a line of smaller slope for all small enough steps.

**Solution:**

**Step 1 (descent lemma along the ray).** $L$-smoothness gives, for every $\alpha \gt 0$,

$$
f(\mathbf{x}_k + \alpha\mathbf{d}_k) \le f(\mathbf{x}_k) + \alpha\,\nabla f_k^T\mathbf{d}_k + \frac{L\alpha^2}{2}\lVert \mathbf{d}_k\rVert^2
$$

**Step 2 (when the model beats the Armijo line).** The Armijo test holds whenever the right side above is
at most $f(\mathbf{x}_k) + c_1\alpha\nabla f_k^T\mathbf{d}_k$, i.e.

$$
(1 - c_1)\,\alpha\,\nabla f_k^T\mathbf{d}_k + \frac{L\alpha^2}{2}\lVert \mathbf{d}_k\rVert^2 \le 0
$$

Dividing by $\alpha \gt 0$ and solving, and writing $\nabla f_k^T\mathbf{d}_k = -\lvert \nabla f_k^T\mathbf{d}_k\rvert$:

$$
\alpha \le \alpha_{\text{safe}} := \frac{2(1-c_1)\,\lvert \nabla f_k^T\mathbf{d}_k\rvert}{L\lVert \mathbf{d}_k\rVert^2}
$$

So **every** $\alpha \in (0, \alpha_{\text{safe}}]$ passes Armijo: the loop cannot run forever, since
$\rho^j\bar{\alpha} \to 0$.

**Step 3 (the floor).** Backtracking accepts the *first* passing value. Either the initial trial
$\bar{\alpha}$ passes (then $\alpha_k = \bar{\alpha}$), or the accepted $\alpha_k = \rho^j\bar{\alpha}$
has predecessor $\alpha_k/\rho$ that failed, hence $\alpha_k/\rho \gt \alpha_{\text{safe}}$, giving

$$
\alpha_k \gt \rho\,\alpha_{\text{safe}} = \frac{2(1-c_1)\rho\,\lvert \nabla f_k^T\mathbf{d}_k\rvert}{L\lVert \mathbf{d}_k\rVert^2}
$$

For steepest descent, $\mathbf{d}_k = -\nabla f_k$ makes
$\lvert \nabla f_k^T\mathbf{d}_k\rvert / \lVert \mathbf{d}_k\rVert^2 = 1$ and the bound reduces to the
clean form quoted.

**Step 4 (guaranteed decrease per step).** Combining the floor with Armijo, for steepest descent,

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - c_1\alpha_k\lVert \nabla f_k\rVert^2 \le f(\mathbf{x}_k) - \frac{2c_1(1-c_1)\rho}{L}\lVert \nabla f_k\rVert^2
$$

which is the sufficient-decrease inequality with an explicit constant — exactly the ingredient the
Zoutendijk argument consumes.

$$
\boxed{\alpha_k \ge \rho\,\frac{2(1-c_1)\lvert \nabla f_k^T\mathbf{d}_k\rvert}{L\lVert \mathbf{d}_k\rVert^2} \ \text{always; for steepest descent } \mathbf{d}_k = -\nabla f_k: \ \alpha_k \ge \min\left\{\bar{\alpha},\ \frac{2(1-c_1)\rho}{L}\right\} \ \Rightarrow \ f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{2c_1(1-c_1)\rho}{L}\lVert \nabla f_k\rVert^2}
$$

> **Key takeaway:** Backtracking never needs to know $L$, yet its output is provably within a factor $\rho$ of the largest theoretically safe step — an adaptive method with a non-adaptive guarantee.

**Verification.** The cell below recomputes every number claimed in Problem L1.2.

In [7]:
# the floor  alpha_k >= min{alpha_bar, rho * 2(1-c1)|g.d| / (L ||d||^2)}  over many directions
Q = np.diag([2.0, 20.0]); L = 20.0            # f = x^2 + 10y^2, so grad f is 20-Lipschitz
f12 = lambda v: 0.5*v @ Q @ v
g12 = lambda v: Q @ v
c1, rho, a_bar = 1e-4, 0.5, 1.0

def backtrack(x, d):
    a = a_bar
    while f12(x + a*d) > f12(x) + c1*a*(g12(x) @ d):
        a *= rho
    return a

worst = np.inf
for _ in range(500):
    x = rng.standard_normal(2)
    d = rng.standard_normal(2)
    if g12(x) @ d >= 0:
        d = -d                                  # make it a descent direction
    a = backtrack(x, d)
    floor = min(a_bar, rho*2*(1-c1)*abs(g12(x) @ d)/(L*(d @ d)))
    worst = min(worst, a/floor)
print(f"over 500 random descent directions, min(alpha_accepted / floor) = {worst:.4f}")
print("steepest-descent specialization: |g.d|/||d||^2 = 1, floor = "
      f"{min(a_bar, rho*2*(1-c1)/L):.6f}")
assert worst >= 1.0

over 500 random descent directions, min(alpha_accepted / floor) = 1.0000
steepest-descent specialization: |g.d|/||d||^2 = 1, floor = 0.049995


### Problem L1.3 — Wolfe Step Lengths Always Exist

**Problem Statement:** Let $\phi(\alpha) = f(\mathbf{x}_k + \alpha\mathbf{d}_k)$ be continuously
differentiable with $\phi'(0) \lt 0$, and suppose $\phi$ is bounded below. Prove that for any
$0 \lt c_1 \lt c_2 \lt 1$ there exists an interval of step lengths satisfying both Wolfe conditions.

*Intuition:* The Armijo line eventually crosses the graph of $\phi$; at the crossing the mean value theorem hands you a point whose slope is exactly $c_1\phi'(0)$, which is already flatter than $c_2\phi'(0)$.

**Solution:**

**Step 1 (the Armijo line is crossed).** Let $\ell(\alpha) = \phi(0) + c_1\alpha\phi'(0)$. Since
$c_1\phi'(0) \lt 0$, $\ell(\alpha) \to -\infty$ as $\alpha \to \infty$, while $\phi$ is bounded below.
Hence $\ell(\alpha) \lt \phi(\alpha)$ for large $\alpha$. Near $0$, however,
$\phi(\alpha) - \ell(\alpha) = (1-c_1)\phi'(0)\alpha + o(\alpha) \lt 0$ for small $\alpha \gt 0$. By the
intermediate value theorem applied to the continuous function $\phi - \ell$, there is a smallest
$\alpha' \gt 0$ with

$$
\phi(\alpha') = \ell(\alpha') = \phi(0) + c_1\alpha'\phi'(0)
$$

and (W1) holds on all of $(0, \alpha']$ by minimality of $\alpha'$.

**Step 2 (the mean value theorem supplies the curvature).** Apply the MVT to $\phi$ on $[0, \alpha']$:
there is $\alpha'' \in (0,\alpha')$ with

$$
\phi'(\alpha'') = \frac{\phi(\alpha') - \phi(0)}{\alpha'} = c_1\phi'(0)
$$

**Step 3 (the curvature condition is satisfied there).** Since $\phi'(0) \lt 0$ and $c_1 \lt c_2$,

$$
\phi'(\alpha'') = c_1\phi'(0) \gt c_2\phi'(0)
$$

so (W2) holds at $\alpha''$; and $\alpha'' \lt \alpha'$ so (W1) holds at $\alpha''$ too. Thus $\alpha''$
satisfies both weak Wolfe conditions. Note also $\lvert \phi'(\alpha'')\rvert = c_1\lvert \phi'(0)\rvert \le c_2\lvert \phi'(0)\rvert$,
so $\alpha''$ satisfies the **strong** Wolfe conditions as well.

**Step 4 (an interval, not a point).** $\phi'$ is continuous, so the strict inequalities of Steps 1–3
persist on a neighbourhood of $\alpha''$: the Wolfe-acceptable set contains an open interval.

$$
\boxed{\exists\, \alpha'' \gt 0 \ \text{with} \ \phi(\alpha'') \le \phi(0) + c_1\alpha''\phi'(0) \ \text{and} \ \lvert \phi'(\alpha'')\rvert \le c_2\lvert \phi'(0)\rvert}
$$

> **Key takeaway:** Existence needs only $\phi$ bounded below along the ray and $c_1 \lt c_2$ — no convexity, no smoothness constant — which is why Wolfe line searches are the default safeguard in general-purpose nonlinear solvers.

**Verification.** The cell below recomputes every number claimed in Problem L1.3.

In [8]:
# a non-convex phi that IS bounded below:  phi(a) = 3 + 0.1 a^2 - sin(a) - 0.5 a
phi  = lambda a: 3.0 + 0.1*a**2 - np.sin(a) - 0.5*a
dphi = lambda a: 0.2*a - np.cos(a) - 0.5
c1, c2 = 0.1, 0.5
print(f"phi(0) = {phi(0.0)},  phi'(0) = {dphi(0.0)} < 0,  phi'' = 0.2 + sin(a) changes sign "
      f"(phi'' at a=4 is {0.2 + np.sin(4.0):.4f}), so phi is non-convex but bounded below")

grid = np.linspace(1e-7, 8.0, 800_001)
gap = phi(grid) - (phi(0.0) + c1*grid*dphi(0.0))
cross = grid[np.argmax(gap >= 0)]                            # smallest alpha' with the gap = 0
sub = grid[grid < cross]
a_dd = sub[np.argmin(np.abs(dphi(sub) - c1*dphi(0.0)))]      # MVT point: phi'(a'') = c1 phi'(0)
print(f"first crossing alpha'  = {cross:.6f}   (gap there = {gap[np.argmax(gap >= 0)]:.2e})")
print(f"MVT point   alpha'' = {a_dd:.6f}: phi'(alpha'') = {dphi(a_dd):.6f}"
      f"  vs c1*phi'(0) = {c1*dphi(0.0):.6f}  vs c2*phi'(0) = {c2*dphi(0.0):.6f}")
W1 = phi(a_dd) <= phi(0.0) + c1*a_dd*dphi(0.0)
W2 = dphi(a_dd) >= c2*dphi(0.0)
strong = abs(dphi(a_dd)) <= c2*abs(dphi(0.0))
print(f"at alpha'': W1={W1}, W2={W2}, strong Wolfe={strong}")
width = np.sum((phi(grid) <= phi(0.0) + c1*grid*dphi(0.0)) & (dphi(grid) >= c2*dphi(0.0)))
print(f"the Wolfe-acceptable set is an interval, not a point: {width} of {grid.size} grid points")
assert W1 and W2 and strong and 0 < a_dd < cross and width > 1000

phi(0) = 3.0,  phi'(0) = -1.5 < 0,  phi'' = 0.2 + sin(a) changes sign (phi'' at a=4 is -0.5568), so phi is non-convex but bounded below
first crossing alpha'  = 3.229170   (gap there = 9.91e-06)
MVT point   alpha'' = 1.600670: phi'(alpha'') = -0.149997  vs c1*phi'(0) = -0.150000  vs c2*phi'(0) = -0.750000
at alpha'': W1=True, W2=True, strong Wolfe=True
the Wolfe-acceptable set is an interval, not a point: 214425 of 800001 grid points


### Problem L1.4 — Newton Solves a Quadratic Exactly, and Is Affine Invariant

**Problem Statement:** (a) Show that on $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A\mathbf{x} - \mathbf{b}^T\mathbf{x}$
with $A \succ 0$, the full Newton step from any starting point lands on the minimizer in one iteration.
(b) Prove that Newton's method is invariant under an invertible affine change of variables
$\mathbf{x} = T\mathbf{y}$, while gradient descent is not.

*Intuition:* Newton minimizes the model exactly; when the model *is* the function, one step suffices — and a change of units cannot alter a model-exact step.

**Solution:**

**(a) One-step exactness.** $\nabla f(\mathbf{x}) = A\mathbf{x} - \mathbf{b}$ and $\nabla^2 f = A$, so

$$
\mathbf{x}_1 = \mathbf{x}_0 - A^{-1}\left(A\mathbf{x}_0 - \mathbf{b}\right) = \mathbf{x}_0 - \mathbf{x}_0 + A^{-1}\mathbf{b} = A^{-1}\mathbf{b} = \mathbf{x}^*
$$

for any $\mathbf{x}_0$ — no dependence on the condition number of $A$ whatsoever.

**(b) Affine invariance.** Let $T$ be invertible and define $\tilde{f}(\mathbf{y}) = f(T\mathbf{y})$. The
chain rule gives

$$
\nabla \tilde{f}(\mathbf{y}) = T^T\nabla f(T\mathbf{y}), \qquad \nabla^2\tilde{f}(\mathbf{y}) = T^T \nabla^2 f(T\mathbf{y})\, T
$$

Hence the Newton direction in $\mathbf{y}$-coordinates is

$$
\tilde{\mathbf{p}} = -\left[T^T H T\right]^{-1} T^T\mathbf{g} = -T^{-1}H^{-1}T^{-T}T^T\mathbf{g} = -T^{-1}H^{-1}\mathbf{g} = T^{-1}\mathbf{p}
$$

where $\mathbf{p}$ is the Newton direction in $\mathbf{x}$-coordinates. So if
$\mathbf{x}_k = T\mathbf{y}_k$ then $\mathbf{x}_{k+1} = T\mathbf{y}_{k+1}$: **the two runs are the same
sequence viewed in two coordinate systems.**

**Gradient descent is not invariant.** Its direction transforms as
$-\nabla\tilde{f}(\mathbf{y}) = -T^T\mathbf{g}$, which equals $T^{-1}(-\mathbf{g})$ only when
$T^T = T^{-1}$, i.e. $T$ orthogonal. Rescaling a single feature by $10^3$ therefore changes the gradient
descent trajectory entirely (and changes $\kappa$), while leaving Newton's trajectory untouched.

$$
\boxed{\text{Newton on a quadratic: } \mathbf{x}_1 = \mathbf{x}^*; \quad \mathbf{p}_{\text{Newton}}(T\mathbf{y}) = T\,\tilde{\mathbf{p}}_{\text{Newton}}(\mathbf{y}) \ \text{for every invertible } T}
$$

> **Key takeaway:** Affine invariance is the deep reason Newton is immune to ill-conditioning: rescaling features is an affine map, and Newton simply does not see it — which is also why its convergence theory is stated with self-concordance rather than with $\kappa$.

**Verification.** The cell below recomputes every number claimed in Problem L1.4.

In [9]:
# (a) one Newton step solves a quadratic exactly; (b) affine invariance under a random T
n = 5
A = rng.standard_normal((n, n)); Qq = A @ A.T + n*np.eye(n)
b = rng.standard_normal(n)
x_star = np.linalg.solve(Qq, b)
x0 = rng.standard_normal(n) * 50.0                       # a deliberately terrible start
x1 = x0 - np.linalg.solve(Qq, Qq @ x0 - b)
print(f"(a) ||x_1 - x*|| after ONE Newton step = {np.linalg.norm(x1 - x_star):.3e}"
      f"   (kappa(Q) = {np.linalg.cond(Qq):.1f})")

f14 = lambda u: np.sum(np.exp(u) - u)
g14 = lambda u: np.exp(u) - 1.0
h14 = lambda u: np.diag(np.exp(u))
T = rng.standard_normal((3, 3)) + 3*np.eye(3)            # invertible, not orthogonal
xk = np.array([0.4, -0.3, 0.25]); yk = np.linalg.solve(T, xk)
gd_x, gd_y = xk.copy(), yk.copy()
for _ in range(4):
    xk = xk - np.linalg.solve(h14(xk), g14(xk))
    yk = yk - np.linalg.solve(T.T @ h14(T @ yk) @ T, T.T @ g14(T @ yk))
    gd_x = gd_x - 0.05*g14(gd_x)
    gd_y = gd_y - 0.05*(T.T @ g14(T @ gd_y))
print(f"(b) Newton:           ||x_4 - T y_4|| = {np.linalg.norm(xk - T @ yk):.3e}")
print(f"    gradient descent: ||x_4 - T y_4|| = {np.linalg.norm(gd_x - T @ gd_y):.3e}")
assert np.linalg.norm(x1 - x_star) < 1e-10
assert np.linalg.norm(xk - T @ yk) < 1e-12
assert np.linalg.norm(gd_x - T @ gd_y) > 1e-3

(a) ||x_1 - x*|| after ONE Newton step = 1.583e-14   (kappa(Q) = 3.6)
(b) Newton:           ||x_4 - T y_4|| = 1.231e-16
    gradient descent: ||x_4 - T y_4|| = 4.178e-01


### Problem L1.5 — Watching the Digits Double

**Problem Statement:** Minimize $f(x) = \frac{x^4}{4} - x$ by Newton's method from $x_0 = 2$. Write the
iteration in closed form, compute four iterates, and verify empirically the quadratic error law
$e_{k+1} \approx C e_k^2$ with $C = \frac{f'''(x^*)}{2f''(x^*)}$.

*Intuition:* Newton for minimization is Newton's root-finder applied to $f'$, and its error constant is the ratio of the second and first derivatives of $f'$ at the root.

**Solution:**

**Step 1 (setup).** $f'(x) = x^3 - 1$, $f''(x) = 3x^2$, $f'''(x) = 6x$. The minimizer is $x^* = 1$, where
$f''(1) = 3 \gt 0$ (SOSC holds). The iteration is

$$
x_{k+1} = x_k - \frac{x_k^3 - 1}{3x_k^2} = \frac{2x_k^3 + 1}{3x_k^2}
$$

**Step 2 (iterates from $x_0 = 2$).**

| $k$ | $x_k$ | $e_k = \lvert x_k - 1\rvert$ | $e_k / e_{k-1}^2$ |
|---|---|---|---|
| $0$ | $2$ | $1$ | — |
| $1$ | $17/12 = 1.4166667$ | $4.16667\times 10^{-1}$ | $0.4167$ |
| $2$ | $1.1105344$ | $1.10534\times 10^{-1}$ | $0.6367$ |
| $3$ | $1.0106368$ | $1.06368\times 10^{-2}$ | $0.8706$ |
| $4$ | $1.0001116$ | $1.11557\times 10^{-4}$ | $0.9860$ |

(The next iterate has $e_5 = 1.2443\times 10^{-8}$, and the one after that $e_6 \approx 1.5\times 10^{-16}$.)

**Step 3 (the predicted constant).** For Newton applied to $g = f'$, the standard error recursion is

$$
e_{k+1} = \frac{\lvert g''(\xi_k)\rvert}{2\lvert g'(x_k)\rvert}e_k^2 \longrightarrow C = \frac{\lvert f'''(x^*)\rvert}{2 f''(x^*)} = \frac{6}{2\cdot 3} = 1
$$

The empirical ratios $0.4167, 0.6367, 0.8706, 0.9860, 0.9999$ climb monotonically toward $1$ exactly as
the iterates enter the asymptotic regime, confirming the law.

**Step 4 (digit count).** Correct digits $-\log_{10}e_k$: $0,\ 0.38,\ 0.96,\ 1.97,\ 3.95,\ 7.91,\ 15.8$ —
doubling once the asymptotic regime is reached at $k \approx 2$.

$$
\boxed{x_{k+1} = \frac{2x_k^3+1}{3x_k^2}, \qquad e_{k+1} \approx \frac{f'''(1)}{2f''(1)}\,e_k^2 = e_k^2}
$$

> **Key takeaway:** The quadratic constant is a *third-derivative-to-second-derivative* ratio: Newton is fast where the Hessian changes slowly relative to its own size, which is precisely what self-concordance formalizes.

**Verification.** The cell below recomputes every number claimed in Problem L1.5.

In [10]:
# Newton on f(x) = x^4/4 - x from x0 = 2:  x_{k+1} = (2x_k^3 + 1)/(3 x_k^2)
x, iters = 2.0, []
for _ in range(6):
    iters.append(x); x = (2*x**3 + 1)/(3*x**2)
iters = np.array(iters)
err = np.abs(iters - 1.0)
ratio = err[1:]/err[:-1]**2
print("k    x_k              e_k              e_k / e_(k-1)^2")
for k in range(len(iters)):
    r = f"{ratio[k-1]:.4f}" if k else "-"
    print(f"{k}    {iters[k]:<16.7f} {err[k]:<16.5e} {r}")
print(f"predicted constant  (3rd deriv)/(2 x 2nd deriv) at x*=1  = {6.0/(2*3.0)}")
print("correct digits -log10 e_k:", np.array2string(-np.log10(err[err > 0]), precision=2))
assert np.isclose(iters[1], 17/12)
assert np.allclose(iters[:5], [2.0, 1.4166667, 1.1105344, 1.0106368, 1.0001116], atol=1e-6)
assert abs(ratio[-1] - 1.0) < 1e-3

k    x_k              e_k              e_k / e_(k-1)^2
0    2.0000000        1.00000e+00      -
1    1.4166667        4.16667e-01      0.4167
2    1.1105344        1.10534e-01      0.6367
3    1.0106368        1.06368e-02      0.8706
4    1.0001116        1.11557e-04      0.9860
5    1.0000000        1.24432e-08      0.9999
predicted constant  (3rd deriv)/(2 x 2nd deriv) at x*=1  = 1.0
correct digits -log10 e_k: [-0.    0.38  0.96  1.97  3.95  7.91]


### Problem L1.6 — The BFGS Update Satisfies the Secant Equation

**Problem Statement:** With $\mathbf{s}_k = \mathbf{x}_{k+1}-\mathbf{x}_k$ and
$\mathbf{y}_k = \nabla f(\mathbf{x}_{k+1}) - \nabla f(\mathbf{x}_k)$, verify that the BFGS update

$$
B_{k+1} = B_k - \frac{B_k\mathbf{s}_k\mathbf{s}_k^T B_k}{\mathbf{s}_k^T B_k\mathbf{s}_k} + \frac{\mathbf{y}_k\mathbf{y}_k^T}{\mathbf{y}_k^T\mathbf{s}_k}
$$

satisfies the secant equation $B_{k+1}\mathbf{s}_k = \mathbf{y}_k$, and explain what the secant equation
means in one dimension.

*Intuition:* The update deletes the old curvature estimate along $\mathbf{s}_k$ and installs the measured one.

**Solution:**

**Step 1 (apply the update to $\mathbf{s}_k$).** Suppress the index $k$:

$$
B_{+}\mathbf{s} = B\mathbf{s} - \frac{B\mathbf{s}\,(\mathbf{s}^T B\mathbf{s})}{\mathbf{s}^T B\mathbf{s}} + \frac{\mathbf{y}\,(\mathbf{y}^T\mathbf{s})}{\mathbf{y}^T\mathbf{s}}
$$

**Step 2 (cancel).** The scalars $\mathbf{s}^T B\mathbf{s}$ and $\mathbf{y}^T\mathbf{s}$ divide out
exactly:

$$
B_{+}\mathbf{s} = B\mathbf{s} - B\mathbf{s} + \mathbf{y} = \mathbf{y}
$$

so the secant equation holds identically — provided the two denominators are nonzero, which the curvature
condition $\mathbf{y}^T\mathbf{s} \gt 0$ and $B \succ 0$ guarantee.

**Step 3 (structure of the update).** The middle term is a rank-one *deflation*: it removes exactly the
component of $B$ acting along $\mathbf{s}$ (note $\frac{B\mathbf{s}\mathbf{s}^TB}{\mathbf{s}^TB\mathbf{s}}$
maps $\mathbf{s} \mapsto B\mathbf{s}$). The last term is a rank-one *insertion* of the freshly measured
curvature $\mathbf{y}$ along that same direction. BFGS is therefore rank-two: it corrects one direction
per iteration, and after $n$ linearly independent steps on a quadratic it reproduces the exact Hessian.

**Step 4 (one dimension).** For $n = 1$ the secant equation reads

$$
B_{+} = \frac{y}{s} = \frac{f'(x_{k+1}) - f'(x_k)}{x_{k+1} - x_k} \approx f''(x_{k+1})
$$

the finite-difference (secant) approximation to the second derivative — hence the name. Quasi-Newton in
$\mathbb{R}^n$ is the multivariate secant method, with the extra $n(n-1)/2$ degrees of freedom in the
symmetric $B_{+}$ fixed by requiring the *least change* from $B_k$ in a weighted Frobenius norm.

$$
\boxed{B_{k+1}\mathbf{s}_k = \mathbf{y}_k \ \text{holds by construction; in 1D it is } B_{+} = \frac{f'(x_{k+1})-f'(x_k)}{x_{k+1}-x_k}}
$$

> **Key takeaway:** BFGS never differentiates twice; it *measures* curvature along the direction it actually travelled and stores that measurement, which is why its cost is first-order while its behaviour is second-order.

**Verification.** The cell below recomputes every number claimed in Problem L1.6.

In [11]:
# B_+ s = y for the direct BFGS update, on random data with s.y > 0
def bfgs_direct(B, s, y):
    Bs = B @ s
    return B - np.outer(Bs, Bs)/(s @ Bs) + np.outer(y, y)/(y @ s)

worst = 0.0
for _ in range(200):
    n = int(rng.integers(2, 7))
    Z = rng.standard_normal((n, n)); B = Z @ Z.T + n*np.eye(n)
    s = rng.standard_normal(n)
    y = (Z @ Z.T + n*np.eye(n)) @ s                     # y = (an SPD matrix) s, so s.y > 0
    Bp = bfgs_direct(B, s, y)
    worst = max(worst, np.linalg.norm(Bp @ s - y)/np.linalg.norm(y))
print(f"max relative secant residual ||B_+ s - y||/||y|| over 200 draws = {worst:.3e}")

s1 = np.array([2.0]); y1 = np.array([0.5])              # the 1-D reading: B_+ = y/s
print(f"1-D case: B_+ = {bfgs_direct(np.array([[3.0]]), s1, y1)[0,0]:.6f}"
      f"   vs  y/s = {(y1/s1)[0]:.6f}")
assert worst < 1e-12
assert np.isclose(bfgs_direct(np.array([[3.0]]), s1, y1)[0, 0], 0.25)

max relative secant residual ||B_+ s - y||/||y|| over 200 draws = 2.748e-16
1-D case: B_+ = 0.250000   vs  y/s = 0.250000


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Newton on Logistic Regression Is IRLS

**Problem Statement:** For logistic regression with labels $y_i \in \{0,1\}$, probabilities
$p_i = \sigma(\mathbf{x}_i^T\mathbf{w})$ and negative log-likelihood
$f(\mathbf{w}) = -\sum_i \left[y_i\log p_i + (1-y_i)\log(1-p_i)\right]$, derive the gradient and Hessian,
write the Newton step, and show it is a weighted least squares solve (iteratively reweighted least
squares).

*Intuition:* Each Newton step refits a *linear* regression on a working response, with weights given by the current predictive variances.

**Solution:**

**Step 1 (gradient).** Using $\sigma'(t) = \sigma(t)(1-\sigma(t))$ and the standard cancellation,

$$
\nabla f(\mathbf{w}) = \sum_{i=1}^m (p_i - y_i)\,\mathbf{x}_i = X^T(\mathbf{p} - \mathbf{y})
$$

**Step 2 (Hessian).** Differentiating once more,

$$
\nabla^2 f(\mathbf{w}) = \sum_{i=1}^m p_i(1-p_i)\,\mathbf{x}_i\mathbf{x}_i^T = X^T S X, \qquad S = \operatorname{diag}\left(p_i(1-p_i)\right)
$$

Since $p_i \in (0,1)$, $S \succ 0$, hence $X^TSX \succeq 0$ and $\succ 0$ whenever $X$ has full column
rank: the objective is convex, so Newton directions are descent directions (Problem L0.3).

**Step 3 (the Newton step).**

$$
\mathbf{w}_{+} = \mathbf{w} - \left(X^TSX\right)^{-1}X^T(\mathbf{p}-\mathbf{y})
$$

**Step 4 (rewrite as weighted least squares).** Factor $X^TSX$ out of both terms:

$$
\mathbf{w}_{+} = \left(X^TSX\right)^{-1}\left[X^TSX\,\mathbf{w} - X^T(\mathbf{p}-\mathbf{y})\right] = \left(X^TSX\right)^{-1}X^TS\left[X\mathbf{w} + S^{-1}(\mathbf{y}-\mathbf{p})\right]
$$

Defining the **working response** $\mathbf{z} = X\mathbf{w} + S^{-1}(\mathbf{y}-\mathbf{p})$,

$$
\mathbf{w}_{+} = \left(X^TSX\right)^{-1}X^TS\,\mathbf{z} = \arg\min_{\mathbf{v}} \ \left(\mathbf{z} - X\mathbf{v}\right)^T S \left(\mathbf{z}-X\mathbf{v}\right)
$$

exactly the normal equations of a weighted least squares problem with weights
$s_i = p_i(1-p_i)$ — the **IRLS** algorithm used by every GLM fitter.

**Step 5 (practical reading).** Well-classified points ($p_i \to 0$ or $1$) get weight $s_i \to 0$ and drop
out; uncertain points ($p_i \approx 1/2$) get the maximal weight $1/4$. Cost per iteration is one
$O(mn^2 + n^3)$ solve, versus $O(mn)$ for a gradient step — worth it when $n$ is moderate, which is why
`statsmodels` and R's `glm` use Newton while deep-learning frameworks do not.

$$
\boxed{\mathbf{w}_{+} = (X^TSX)^{-1}X^TS\mathbf{z}, \quad S = \operatorname{diag}(p_i(1-p_i)), \quad \mathbf{z} = X\mathbf{w} + S^{-1}(\mathbf{y}-\mathbf{p})}
$$

> **Key takeaway:** Newton's method on a generalized linear model *is* iteratively reweighted least squares — the algebra of second-order optimization and the statistics of Fisher scoring are the same computation.

**Verification.** The cell below recomputes every number claimed in Problem L2.1.

In [12]:
# Newton on logistic regression == IRLS: compare the Newton step with a weighted-least-squares solve
m_obs, n_dim = 200, 4
X = rng.standard_normal((m_obs, n_dim))
w_true = rng.standard_normal(n_dim)
y_lab = (rng.random(m_obs) < 1/(1 + np.exp(-X @ w_true))).astype(float)
sigmoid = lambda t: 1/(1 + np.exp(-t))

w = np.zeros(n_dim)
for it in range(8):
    p = sigmoid(X @ w)
    S = p*(1 - p)
    grad = X.T @ (p - y_lab)
    Hess = X.T @ (S[:, None]*X)
    w_newton = w - np.linalg.solve(Hess, grad)
    z = X @ w + (y_lab - p)/S                                  # working response
    w_irls = np.linalg.solve(X.T @ (S[:, None]*X), X.T @ (S*z))  # weighted least squares
    print(f"it {it}: ||grad|| = {np.linalg.norm(grad):.3e}   "
          f"||w_newton - w_irls|| = {np.linalg.norm(w_newton - w_irls):.3e}")
    assert np.linalg.norm(w_newton - w_irls) < 1e-9
    w = w_newton
print(f"weights S = p(1-p) lie in [{S.min():.4f}, {S.max():.4f}]  (max possible 0.25)")
assert np.linalg.norm(X.T @ (sigmoid(X @ w) - y_lab)) < 1e-8
assert S.max() <= 0.25 + 1e-12

it 0: ||grad|| = 6.372e+01   ||w_newton - w_irls|| = 0.000e+00
it 1: ||grad|| = 1.452e+01   ||w_newton - w_irls|| = 6.798e-16
it 2: ||grad|| = 3.300e+00   ||w_newton - w_irls|| = 7.417e-16
it 3: ||grad|| = 3.116e-01   ||w_newton - w_irls|| = 1.437e-15
it 4: ||grad|| = 3.512e-03   ||w_newton - w_irls|| = 1.157e-15
it 5: ||grad|| = 4.577e-07   ||w_newton - w_irls|| = 1.337e-15
it 6: ||grad|| = 5.930e-15   ||w_newton - w_irls|| = 4.719e-15
it 7: ||grad|| = 3.834e-15   ||w_newton - w_irls|| = 4.856e-15
weights S = p(1-p) lie in [0.0029, 0.2500]  (max possible 0.25)


### Problem L2.2 — A Newton Step That Increases the Rosenbrock Objective

**Problem Statement:** At $\mathbf{x}_0 = (0,0)$ on the Rosenbrock function
$f(x,y) = (1-x)^2 + 100(y-x^2)^2$, compute the Newton direction, verify it is a descent direction, and
show that the *full* Newton step increases $f$. Then find what a backtracking line search does instead.

*Intuition:* The quadratic model is trustworthy only inside a small ball; Rosenbrock's fourth-order term punishes any step that leaves it.

**Solution:**

**Step 1 (derivatives at the origin).**

$$
f_x = -2(1-x) - 400x(y-x^2), \qquad f_y = 200(y-x^2)
$$

so $\mathbf{g}_0 = \nabla f(0,0) = (-2, 0)$ and $f(0,0) = 1$. The Hessian is

$$
\nabla^2 f = \begin{bmatrix} 2 - 400y + 1200x^2 & -400x \\ -400x & 200\end{bmatrix}, \qquad H_0 = \begin{bmatrix} 2 & 0 \\ 0 & 200\end{bmatrix} \succ 0
$$

**Step 2 (Newton direction).**

$$
\mathbf{p}_0 = -H_0^{-1}\mathbf{g}_0 = -\begin{bmatrix} 1/2 & 0 \\ 0 & 1/200\end{bmatrix}\begin{pmatrix} -2 \\ 0\end{pmatrix} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}
$$

It **is** a descent direction: $\mathbf{g}_0^T\mathbf{p}_0 = -2 \lt 0$, as guaranteed by $H_0 \succ 0$.

**Step 3 (the full step overshoots badly).** $\mathbf{x}_0 + \mathbf{p}_0 = (1, 0)$ and

$$
f(1,0) = (1-1)^2 + 100(0 - 1)^2 = 100 \ \gg \ 1 = f(0,0)
$$

The model predicted $m_0(\mathbf{p}_0) = 1 + (-2)(1) + \frac{1}{2}(1)(2)(1) = 0$; the truth is $100$. The
culprit is the quartic term $100x^4$, which the quadratic model cannot see.

**Step 4 (what backtracking does).** With $c_1 = 10^{-4}$, the Armijo test on
$\phi(\alpha) = f(\alpha, 0) = (1-\alpha)^2 + 100\alpha^4$ requires
$\phi(\alpha) \le 1 - 2\times 10^{-4}\alpha$:

| $\alpha$ | $\phi(\alpha) = (1-\alpha)^2 + 100\alpha^4$ | bound | verdict |
|---|---|---|---|
| $1$ | $0 + 100 = 100$ | $0.9998$ | reject |
| $0.5$ | $0.25 + 6.25 = 6.5$ | $0.9999$ | reject |
| $0.25$ | $0.5625 + 0.3906 = 0.9531$ | $0.99995$ | **accept** |

so the safeguarded step is $\alpha_0 = 1/4$, landing at $(0.25, 0)$ with $f = 0.9531$ — a genuine, if
modest, decrease.

$$
\boxed{\mathbf{p}_0 = (1,0) \text{ is descent, yet } f(\mathbf{x}_0 + \mathbf{p}_0) = 100 \gg f(\mathbf{x}_0) = 1; \ \text{backtracking accepts } \alpha_0 = \tfrac14}
$$

> **Key takeaway:** "Descent direction" and "good step" are different claims; the line search is not an optimization of the step length so much as an insurance policy against the model being wrong at distance $1$.

**Verification.** The cell below recomputes every number claimed in Problem L2.2.

In [13]:
# Rosenbrock at the origin: descent direction, disastrous full step, backtracking rescue
rosen  = lambda v: (1 - v[0])**2 + 100*(v[1] - v[0]**2)**2
rosen_g = lambda v: np.array([-2*(1 - v[0]) - 400*v[0]*(v[1] - v[0]**2), 200*(v[1] - v[0]**2)])
rosen_H = lambda v: np.array([[2 - 400*v[1] + 1200*v[0]**2, -400*v[0]], [-400*v[0], 200.0]])

x0 = np.zeros(2); g0 = rosen_g(x0); H0 = rosen_H(x0)
p0 = -np.linalg.solve(H0, g0)
print(f"g0 = {g0},  H0 =\n{H0}\np0 = {p0},  g0.p0 = {g0 @ p0:+.4f} < 0 (descent)")
print(f"f(x0) = {rosen(x0)},  f(x0 + p0) = {rosen(x0 + p0)}  <- the FULL step increases f")
model = rosen(x0) + g0 @ p0 + 0.5*p0 @ H0 @ p0
print(f"quadratic model predicted {model:.4f}, truth is {rosen(x0 + p0):.4f}")

c1, a = 1e-4, 1.0
while rosen(x0 + a*p0) > rosen(x0) + c1*a*(g0 @ p0):
    print(f"  alpha={a:<7.4f} phi={rosen(x0 + a*p0):<10.6f} "
          f"bound={rosen(x0) + c1*a*(g0 @ p0):<10.6f} reject")
    a *= 0.5
print(f"  alpha={a:<7.4f} phi={rosen(x0 + a*p0):<10.6f} "
      f"bound={rosen(x0) + c1*a*(g0 @ p0):<10.6f} ACCEPT")
assert np.allclose(p0, [1.0, 0.0]) and g0 @ p0 < 0
assert np.isclose(rosen(x0 + p0), 100.0) and np.isclose(model, 0.0)
assert a == 0.25 and np.isclose(rosen(x0 + a*p0), 0.953125)

g0 = [-2.  0.],  H0 =
[[  2.  -0.]
 [ -0. 200.]]
p0 = [ 1. -0.],  g0.p0 = -2.0000 < 0 (descent)
f(x0) = 1.0,  f(x0 + p0) = 100.0  <- the FULL step increases f
quadratic model predicted 0.0000, truth is 100.0000
  alpha=1.0000  phi=100.000000 bound=0.999800   reject
  alpha=0.5000  phi=6.500000   bound=0.999900   reject
  alpha=0.2500  phi=0.953125   bound=0.999950   ACCEPT


### Problem L2.3 — L-BFGS Memory Arithmetic at Scale

**Problem Statement:** A model has $n = 10^6$ parameters. Compare the storage of (i) the exact Hessian,
(ii) a dense BFGS approximation, (iii) L-BFGS with memory $m = 10$, all in float64. Then give the per-
iteration flop count of the two-loop recursion and compare it to a gradient step.

*Intuition:* Second-order information costs $O(n^2)$ to store; the whole point of limited memory is to keep only the $2m$ directions you actually walked.

**Solution:**

**Step 1 (dense storage).** A symmetric $n\times n$ matrix in float64 needs $8n^2$ bytes (or half that
exploiting symmetry):

$$
8 \times (10^6)^2 = 8\times 10^{12}\ \text{bytes} = 8\ \text{TB}
$$

Both the exact Hessian (ii is the same size as i) and dense BFGS are therefore completely out of reach —
before even considering the $O(n^3) = 10^{18}$ flops of a factorization.

**Step 2 (L-BFGS storage).** L-BFGS stores $m$ pairs $(\mathbf{s}_i, \mathbf{y}_i)$, each two vectors of
length $n$, plus the scalars $\rho_i = 1/(\mathbf{y}_i^T\mathbf{s}_i)$:

$$
8 \times 2mn = 8 \times 2 \times 10 \times 10^6 = 1.6\times 10^8\ \text{bytes} = 160\ \text{MB}
$$

a factor $\frac{n}{2m} = \frac{10^6}{20} = 5\times 10^4$ smaller than dense BFGS.

**Step 3 (flop count of the two-loop recursion).** The recursion is

- **First loop** ($i = k-1$ down to $k-m$): $\alpha_i = \rho_i\mathbf{s}_i^T\mathbf{q}$ (one dot product,
  $2n$ flops), $\mathbf{q} \leftarrow \mathbf{q} - \alpha_i\mathbf{y}_i$ (one axpy, $2n$ flops).
- **Scaling:** $\mathbf{r} = \gamma_k\mathbf{q}$ with
  $\gamma_k = \frac{\mathbf{s}_{k-1}^T\mathbf{y}_{k-1}}{\mathbf{y}_{k-1}^T\mathbf{y}_{k-1}}$ ($O(n)$).
- **Second loop** ($i = k-m$ up to $k-1$): $\beta = \rho_i\mathbf{y}_i^T\mathbf{r}$ and
  $\mathbf{r} \leftarrow \mathbf{r} + (\alpha_i - \beta)\mathbf{s}_i$ (another $4n$ flops).

Total: $4mn$ flops in each loop pair, i.e.

$$
\approx 8mn = 8\times 10\times 10^6 = 8\times 10^7 \ \text{flops per iteration}
$$

versus $O(n) = 10^6$ for a plain gradient step: about $80\times$ the vector work, but *negligible*
compared to the cost of the gradient evaluation itself in any realistic model.

$$
\boxed{\text{dense BFGS: } 8\ \text{TB}; \quad \text{L-BFGS}(m{=}10): 160\ \text{MB} \ \text{and} \approx 8mn = 8\times 10^7 \ \text{flops/iteration}}
$$

> **Key takeaway:** L-BFGS never forms a matrix — it applies the *operator* $H_k^{-1}$ through a sequence of dot products and axpys, which is the reason second-order optimization survives at all in the million-parameter regime.

**Verification.** The cell below recomputes every number claimed in Problem L2.3.

In [14]:
# storage and flop arithmetic for n = 1e6 parameters
n, m_mem, bytes_f64 = 10**6, 10, 8
dense = bytes_f64 * n**2
lbfgs = bytes_f64 * 2 * m_mem * n
print(f"dense Hessian / dense BFGS : {dense:.3e} bytes = {dense/1e12:.1f} TB")
print(f"L-BFGS with m = {m_mem}        : {lbfgs:.3e} bytes = {lbfgs/1e6:.0f} MB")
print(f"ratio                      : {dense/lbfgs:.3e}  (= n/(2m) = {n/(2*m_mem):.1e})")
flops_two_loop = 8*m_mem*n          # 4n per pair in each of the two loops
print(f"two-loop flops per iteration: {flops_two_loop:.3e}  "
      f"vs O(n) = {n:.3e} for a plain gradient step  ({flops_two_loop/n:.0f}x)")
print(f"a dense Cholesky would cost n^3/3 = {n**3/3:.3e} flops")
assert dense == 8e12 and lbfgs == 1.6e8 and flops_two_loop == 8e7
assert abs(dense/lbfgs - n/(2*m_mem)) < 1e-6

dense Hessian / dense BFGS : 8.000e+12 bytes = 8.0 TB
L-BFGS with m = 10        : 1.600e+08 bytes = 160 MB
ratio                      : 5.000e+04  (= n/(2m) = 5.0e+04)
two-loop flops per iteration: 8.000e+07  vs O(n) = 1.000e+06 for a plain gradient step  (80x)
a dense Cholesky would cost n^3/3 = 3.333e+17 flops


### Problem L2.4 — The Two-Loop Recursion, Derived and Executed

**Problem Statement:** Show that the inverse-Hessian BFGS update
$H_{k+1} = V_k^T H_k V_k + \rho_k\mathbf{s}_k\mathbf{s}_k^T$ with
$V_k = I - \rho_k\mathbf{y}_k\mathbf{s}_k^T$, $\rho_k = 1/(\mathbf{y}_k^T\mathbf{s}_k)$, unrolls into the
two-loop recursion. Then execute one L-BFGS step with $m = 1$, $\mathbf{s} = (1,0)^T$,
$\mathbf{y} = (2,0)^T$, $H^0 = I$, $\mathbf{g} = (1,1)^T$.

*Intuition:* Unrolling the recursion $m$ times expresses $H_k\mathbf{g}$ as sandwiched products of rank-one factors — and each sandwich layer is one pass of a loop.

**Solution:**

**Step 1 (unroll).** Applying the recursion $m$ times from an initial $H_k^0$,

$$
H_k = \left(V_{k-1}^T\cdots V_{k-m}^T\right)H_k^0\left(V_{k-m}\cdots V_{k-1}\right) + \sum_{i=k-m}^{k-1}\rho_i\left(V_{k-1}^T\cdots V_{i+1}^T\right)\mathbf{s}_i\mathbf{s}_i^T\left(V_{i+1}\cdots V_{k-1}\right)
$$

The product $\mathbf{q} = V_{k-m}\cdots V_{k-1}\mathbf{g}$ is computed right-to-left, and because
$V_i\mathbf{q} = \mathbf{q} - \rho_i\mathbf{y}_i(\mathbf{s}_i^T\mathbf{q}) = \mathbf{q} - \alpha_i\mathbf{y}_i$
with $\alpha_i = \rho_i\mathbf{s}_i^T\mathbf{q}$, this is exactly the **first loop**, which also stores the
$\alpha_i$. Applying $H_k^0$ is the middle scaling. Multiplying back by the $V_i^T$ and adding the
$\rho_i\mathbf{s}_i\mathbf{s}_i^T$ terms — whose contribution at layer $i$ is precisely
$(\alpha_i - \beta_i)\mathbf{s}_i$ with $\beta_i = \rho_i\mathbf{y}_i^T\mathbf{r}$ — is the **second loop**.

**Step 2 (execute with $m = 1$).** Data: $\mathbf{s} = (1,0)^T$, $\mathbf{y} = (2,0)^T$,
$\mathbf{y}^T\mathbf{s} = 2 \gt 0$ so $\rho = 1/2$; $\mathbf{q} = \mathbf{g} = (1,1)^T$.

*First loop:*

$$
\alpha = \rho\,\mathbf{s}^T\mathbf{q} = \tfrac{1}{2}(1) = \tfrac{1}{2}, \qquad \mathbf{q} \leftarrow \mathbf{q} - \alpha\mathbf{y} = (1,1)^T - \tfrac12(2,0)^T = (0,1)^T
$$

*Scaling:* the standard choice is
$\gamma = \frac{\mathbf{s}^T\mathbf{y}}{\mathbf{y}^T\mathbf{y}} = \frac{2}{4} = \frac12$, so
$\mathbf{r} = \gamma\mathbf{q} = (0, 0.5)^T$.

*Second loop:*

$$
\beta = \rho\,\mathbf{y}^T\mathbf{r} = \tfrac12 (0) = 0, \qquad \mathbf{r} \leftarrow \mathbf{r} + (\alpha - \beta)\mathbf{s} = (0,0.5)^T + \tfrac12(1,0)^T = (0.5,\ 0.5)^T
$$

**Step 3 (the step).** The search direction is $\mathbf{d} = -\mathbf{r} = (-0.5, -0.5)^T$, and
$\mathbf{g}^T\mathbf{d} = -1 \lt 0$: a descent direction, as L-BFGS guarantees whenever every stored pair
has $\mathbf{y}_i^T\mathbf{s}_i \gt 0$.

**Sanity check.** Along the measured direction $\mathbf{s} = \mathbf{e}_1$ the observed curvature is
$y_1/s_1 = 2$, so the model applies the inverse curvature $1/2$ to the first component — and indeed
$r_1 = 0.5 = g_1/2$. Along the *unmeasured* direction $\mathbf{e}_2$ it falls back on the scaling
$\gamma = 1/2$.

$$
\boxed{\mathbf{d} = -H\mathbf{g} = (-0.5,\ -0.5)^T \ \text{computed in } O(mn) \text{ without forming } H}
$$

> **Key takeaway:** The two-loop recursion is nothing but the associativity of rank-one products used in the right order — the algorithmic payoff of writing the BFGS inverse update in the sandwiched form $V^THV + \rho\mathbf{s}\mathbf{s}^T$.

**Verification.** The cell below recomputes every number claimed in Problem L2.4.

In [15]:
# the two-loop recursion with m = 1: s = (1,0), y = (2,0), H^0 = gamma I, g = (1,1)
def two_loop(g, pairs, gamma):
    q, alphas = g.copy(), []
    for s, y in reversed(pairs):
        a = (s @ q)/(y @ s); alphas.append(a); q = q - a*y
    r = gamma*q
    for (s, y), a in zip(pairs, reversed(alphas)):
        b = (y @ r)/(y @ s); r = r + (a - b)*s
    return r

s = np.array([1.0, 0.0]); y = np.array([2.0, 0.0]); g = np.array([1.0, 1.0])
gamma = (s @ y)/(y @ y)
r = two_loop(g, [(s, y)], gamma)
print(f"rho = {1/(y @ s)},  gamma = s.y/y.y = {gamma}")
print(f"H g = {r},   search direction d = {-r},   g.d = {g @ (-r):+.4f}")

I = np.eye(2); rho = 1/(y @ s)
H_dense = (I - rho*np.outer(s, y)) @ (gamma*I) @ (I - rho*np.outer(y, s)) + rho*np.outer(s, s)
print(f"dense H = \n{H_dense}\nresidual ||two_loop - H g|| = {np.linalg.norm(r - H_dense @ g):.3e}")
assert np.allclose(r, [0.5, 0.5]) and g @ (-r) < 0
assert np.linalg.norm(r - H_dense @ g) < 1e-14
assert np.isclose(r[0], g[0]/(y[0]/s[0]))     # inverse of the MEASURED curvature along e1

rho = 0.5,  gamma = s.y/y.y = 0.5
H g = [0.5 0.5],   search direction d = [-0.5 -0.5],   g.d = -1.0000
dense H = 
[[0.5 0. ]
 [0.  0.5]]
residual ||two_loop - H g|| = 0.000e+00


### Problem L2.5 — Gauss-Newton and Levenberg-Marquardt for Nonlinear Least Squares

**Problem Statement:** For $f(\mathbf{x}) = \frac{1}{2}\lVert \mathbf{r}(\mathbf{x})\rVert^2$ with residual
map $\mathbf{r}: \mathbb{R}^n \to \mathbb{R}^m$ and Jacobian $J$, derive the exact Hessian, state the
Gauss-Newton approximation and when it is accurate, and show that Levenberg-Marquardt interpolates between
Gauss-Newton and gradient descent.

*Intuition:* Dropping the curvature-of-the-residual term costs nothing near a good fit, because that term is multiplied by the residuals themselves.

**Solution:**

**Step 1 (exact derivatives).** With $\mathbf{r} = (r_1,\dots,r_m)^T$ and $J_{ij} = \partial r_i/\partial x_j$,

$$
\nabla f(\mathbf{x}) = J^T\mathbf{r}, \qquad \nabla^2 f(\mathbf{x}) = J^TJ + \sum_{i=1}^m r_i\,\nabla^2 r_i(\mathbf{x})
$$

**Step 2 (Gauss-Newton).** Drop the second term:

$$
H_{\text{GN}} = J^TJ \succeq 0, \qquad \mathbf{p}_{\text{GN}} = -\left(J^TJ\right)^{-1}J^T\mathbf{r}
$$

This is exactly the normal-equation solution of the *linearized* least squares problem
$\min_{\mathbf{p}} \lVert \mathbf{r} + J\mathbf{p}\rVert^2$, so no second derivatives are ever needed.

**Step 3 (when it is accurate).** The dropped term is $\sum_i r_i \nabla^2 r_i$, whose size is
$O(\lVert \mathbf{r}\rVert \cdot \max_i \lVert \nabla^2 r_i\rVert)$. Two regimes:

- **Small residual** ($\mathbf{r}(\mathbf{x}^*) \approx \mathbf{0}$, a nearly exact fit): the dropped term
  vanishes at the solution and Gauss-Newton inherits Newton's *quadratic* local rate.
- **Large residual** or strongly curved residuals: the neglected term is $O(1)$ and Gauss-Newton degrades
  to linear convergence, sometimes failing to converge at all.

**Step 4 (Levenberg-Marquardt).** Solve instead

$$
\left(J^TJ + \tau I\right)\mathbf{p}_{\text{LM}} = -J^T\mathbf{r}, \qquad \tau \ge 0
$$

- $\tau \to 0$: $\mathbf{p}_{\text{LM}} \to \mathbf{p}_{\text{GN}}$ — fast, model-trusting steps.
- $\tau \to \infty$: $\mathbf{p}_{\text{LM}} \approx -\frac{1}{\tau}J^T\mathbf{r} = -\frac{1}{\tau}\nabla f$
  — a short *gradient descent* step, always safe.

Moreover $J^TJ + \tau I \succ 0$ for $\tau \gt 0$ even when $J$ is rank deficient, so the direction is
always well-defined and descent. LM is precisely a trust-region method: $\mathbf{p}_{\text{LM}}$ solves
$\min \lVert \mathbf{r} + J\mathbf{p}\rVert^2$ subject to $\lVert \mathbf{p}\rVert \le \Delta$, with $\tau$
the multiplier of the trust-region constraint, and codes adapt $\tau$ by comparing predicted to actual
decrease.

$$
\boxed{\nabla^2 f = J^TJ + \sum_i r_i\nabla^2 r_i; \quad \mathbf{p}_{\text{LM}} = -\left(J^TJ + \tau I\right)^{-1}J^T\mathbf{r} \ \text{interpolates GN} \leftrightarrow \text{GD}}
$$

> **Key takeaway:** Gauss-Newton is second-order accuracy purchased with first-order derivatives, valid exactly when the model fits well — and Levenberg-Marquardt's damping parameter is the same $\tau$ that appears in modified Newton, wearing a trust-region hat.

**Verification.** The cell below recomputes every number claimed in Problem L2.5.

In [16]:
# Gauss-Newton vs the exact Hessian, and Levenberg-Marquardt interpolating GN <-> GD
t_data = np.linspace(0.0, 3.0, 12)
resid = lambda x: x[0]*np.exp(-x[1]*t_data) - (2.0*np.exp(-0.8*t_data) + 0.4)   # large residual
jac   = lambda x: np.column_stack([np.exp(-x[1]*t_data), -x[0]*t_data*np.exp(-x[1]*t_data)])
f25   = lambda x: 0.5*np.sum(resid(x)**2)

x = np.array([1.5, 0.5])
J = jac(x); r = resid(x)
H_gn = J.T @ J
eps = 1e-5                                   # exact Hessian by central differences
H_ex = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        e_i = np.zeros(2); e_i[i] = eps
        e_j = np.zeros(2); e_j[j] = eps
        H_ex[i, j] = (f25(x+e_i+e_j) - f25(x+e_i-e_j) - f25(x-e_i+e_j) + f25(x-e_i-e_j))/(4*eps*eps)
num_grad = np.array([(f25(x + d) - f25(x - d))/(2*eps)
                     for d in (np.array([eps, 0.0]), np.array([0.0, eps]))])
print(f"grad check ||J^T r - numerical grad|| = {np.linalg.norm(J.T @ r - num_grad):.3e}")
print(f"||exact Hessian - J^T J|| = {np.linalg.norm(H_ex - H_gn):.4f}   "
      f"(the dropped term sum r_i grad^2 r_i);  ||r|| = {np.linalg.norm(r):.4f}")

for tau in [0.0, 1.0, 1e4]:
    p = -np.linalg.solve(H_gn + tau*np.eye(2), J.T @ r)
    print(f"  tau={tau:<8.0f} p_LM = {np.array2string(p, precision=5)}   "
          f"angle to -grad = {np.degrees(np.arccos(p @ (-J.T @ r)/(np.linalg.norm(p)*np.linalg.norm(J.T @ r)))):.2f} deg")
p_big = -np.linalg.solve(H_gn + 1e4*np.eye(2), J.T @ r)
assert np.linalg.norm(p_big - (-J.T @ r)/1e4) < 1e-6      # tau -> inf gives a scaled gradient step
assert np.min(np.linalg.eigvalsh(H_gn + 1e-6*np.eye(2))) > 0
assert np.linalg.norm(H_ex - H_gn) > 1e-2                  # large residual: GN is NOT the Hessian

grad check ||J^T r - numerical grad|| = 8.048e-10
||exact Hessian - J^T J|| = 5.9157   (the dropped term sum r_i grad^2 r_i);  ||r|| = 1.5530
  tau=0        p_LM = [0.80739 0.03756]   angle to -grad = 49.06 deg
  tau=1        p_LM = [ 0.551   -0.07012]   angle to -grad = 39.15 deg
  tau=10000    p_LM = [ 0.00031 -0.00032]   angle to -grad = 0.02 deg


### Problem L2.6 — The Newton Decrement as an Affine-Invariant Stopping Rule

**Problem Statement:** Define the Newton decrement
$\lambda(\mathbf{x}) = \left(\nabla f(\mathbf{x})^T \left[\nabla^2 f(\mathbf{x})\right]^{-1}\nabla f(\mathbf{x})\right)^{1/2}$
for $\nabla^2 f \succ 0$. Show that (a) $\lambda^2/2$ estimates the suboptimality $f(\mathbf{x}) - f^*$,
(b) $\lambda^2 = -\nabla f^T\mathbf{p}_{\text{Newton}}$, and (c) $\lambda$ is affine invariant, unlike
$\lVert \nabla f\rVert$.

*Intuition:* The gradient norm depends on your choice of units; the decrement measures the gradient in the units the Hessian itself supplies.

**Solution:**

**(a) Suboptimality estimate.** Let $\hat{f}(\mathbf{y}) = f(\mathbf{x}) + \nabla f^T(\mathbf{y}-\mathbf{x}) + \frac12(\mathbf{y}-\mathbf{x})^TH(\mathbf{y}-\mathbf{x})$
be the second-order model. Its minimizer is $\mathbf{y} = \mathbf{x} + \mathbf{p}$ with
$\mathbf{p} = -H^{-1}\nabla f$, and

$$
\min_{\mathbf{y}}\hat{f}(\mathbf{y}) = f(\mathbf{x}) - \frac{1}{2}\nabla f^T H^{-1}\nabla f = f(\mathbf{x}) - \frac{\lambda^2}{2}
$$

So $f(\mathbf{x}) - \min\hat{f} = \lambda^2/2$: the model's predicted gap. For self-concordant $f$ one has
the rigorous bound $f(\mathbf{x}) - f^* \le \lambda^2$ once $\lambda \le 0.68$, which is why solvers stop
on $\lambda^2/2 \le \epsilon$.

**(b) Directional slope.**

$$
-\nabla f^T\mathbf{p} = -\nabla f^T\left(-H^{-1}\nabla f\right) = \nabla f^T H^{-1}\nabla f = \lambda^2
$$

so $\lambda^2$ is exactly the (magnitude of the) initial slope along the Newton direction — which also
re-proves that $\mathbf{p}$ is a descent direction when $H \succ 0$. Equivalently
$\lambda = \lVert \mathbf{p}\rVert_H$, the Newton step measured in the Hessian norm.

**(c) Affine invariance.** Under $\mathbf{x} = T\mathbf{y}$ with $T$ invertible,
$\tilde{\mathbf{g}} = T^T\mathbf{g}$ and $\tilde{H} = T^THT$ (Problem L1.4), so

$$
\tilde{\lambda}^2 = \tilde{\mathbf{g}}^T\tilde{H}^{-1}\tilde{\mathbf{g}} = \mathbf{g}^TT\left(T^THT\right)^{-1}T^T\mathbf{g} = \mathbf{g}^T T T^{-1}H^{-1}T^{-T}T^T\mathbf{g} = \mathbf{g}^TH^{-1}\mathbf{g} = \lambda^2
$$

By contrast $\lVert \tilde{\mathbf{g}}\rVert = \lVert T^T\mathbf{g}\rVert$ changes with $T$: rescaling one
feature by $10^3$ multiplies that gradient component by $10^3$, so a threshold on
$\lVert \nabla f\rVert$ is a statement about your units, not about your solution.

$$
\boxed{\lambda^2 = \nabla f^TH^{-1}\nabla f = -\nabla f^T\mathbf{p}_{\text{Newton}}; \quad \lambda \text{ is affine invariant, } \lVert \nabla f\rVert \text{ is not}}
$$

> **Key takeaway:** Choose stopping criteria that are invariant under the transformations your algorithm is invariant under — for Newton that means the decrement, which is why interior-point solvers report $\lambda^2/2$ rather than a raw gradient norm.

**Verification.** The cell below recomputes every number claimed in Problem L2.6.

In [17]:
# Newton decrement: lambda^2 = -grad f . p, affine invariant; ||grad f|| is not
f26 = lambda u: np.sum(np.exp(u) - u)
g26 = lambda u: np.exp(u) - 1.0
h26 = lambda u: np.diag(np.exp(u))

x = np.array([0.7, -0.4, 0.3])
g, H = g26(x), h26(x)
lam2 = g @ np.linalg.solve(H, g)
p = -np.linalg.solve(H, g)
print(f"lambda^2 = {lam2:.8f},  -grad.p = {-g @ p:.8f},  ||p||_H^2 = {p @ H @ p:.8f}")
model_gap = f26(x) - (f26(x) + g @ p + 0.5*p @ H @ p)
print(f"model gap f(x) - min model = {model_gap:.8f}  vs lambda^2/2 = {lam2/2:.8f}")

T = rng.standard_normal((3, 3)) + 3*np.eye(3)
y = np.linalg.solve(T, x)
g_t, H_t = T.T @ g26(T @ y), T.T @ h26(T @ y) @ T
lam2_t = g_t @ np.linalg.solve(H_t, g_t)
print(f"after x = T y:  lambda^2 = {lam2_t:.8f} (unchanged),  "
      f"||grad|| {np.linalg.norm(g):.4f} -> {np.linalg.norm(g_t):.4f} (changed)")
assert abs(lam2 - (-g @ p)) < 1e-12 and abs(lam2 - p @ H @ p) < 1e-12
assert abs(model_gap - lam2/2) < 1e-12
assert abs(lam2 - lam2_t) < 1e-10
assert abs(np.linalg.norm(g) - np.linalg.norm(g_t)) > 1e-3

lambda^2 = 0.76315978,  -grad.p = 0.76315978,  ||p||_H^2 = 0.76315978
model gap f(x) - min model = 0.38157989  vs lambda^2/2 = 0.38157989
after x = T y:  lambda^2 = 0.76315978 (unchanged),  ||grad|| 1.1220 -> 4.9556 (changed)


## L3 — Challenge Proofs

### Problem L3.1 — Local Quadratic Convergence of Newton's Method

**Problem Statement:** Let $f \in \mathcal{C}^2$ near $\mathbf{x}^*$ with $\nabla f(\mathbf{x}^*) = \mathbf{0}$,
$\nabla^2 f(\mathbf{x}^*) \succ 0$, and $\nabla^2 f$ Lipschitz with constant $M$ in a neighbourhood of
$\mathbf{x}^*$. Prove that for $\mathbf{x}_0$ close enough, the pure Newton iterates satisfy
$\lVert \mathbf{x}_{k+1}-\mathbf{x}^*\rVert \le C\lVert \mathbf{x}_k - \mathbf{x}^*\rVert^2$ and identify $C$.

*Intuition:* The Newton step cancels the gradient exactly to first order; what remains is the Taylor remainder, and Lipschitz Hessians make that remainder quadratic.

**Solution:**

**Step 1 (set up the error).** Write $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^*$, $H_k = \nabla^2 f(\mathbf{x}_k)$,
$\mathbf{g}_k = \nabla f(\mathbf{x}_k)$. Then

$$
\mathbf{e}_{k+1} = \mathbf{x}_k - H_k^{-1}\mathbf{g}_k - \mathbf{x}^* = H_k^{-1}\left(H_k\mathbf{e}_k - \mathbf{g}_k\right)
$$

**Step 2 (Taylor with integral remainder).** Since $\nabla f(\mathbf{x}^*) = \mathbf{0}$,

$$
\mathbf{g}_k = \nabla f(\mathbf{x}_k) - \nabla f(\mathbf{x}^*) = \int_0^1 \nabla^2 f\left(\mathbf{x}^* + t\mathbf{e}_k\right)\mathbf{e}_k\, dt
$$

Therefore

$$
H_k\mathbf{e}_k - \mathbf{g}_k = \int_0^1 \left[\nabla^2 f(\mathbf{x}_k) - \nabla^2 f(\mathbf{x}^* + t\mathbf{e}_k)\right]\mathbf{e}_k\, dt
$$

**Step 3 (bound with the Lipschitz constant).** For each $t$,
$\lVert \mathbf{x}_k - (\mathbf{x}^* + t\mathbf{e}_k)\rVert = (1-t)\lVert \mathbf{e}_k\rVert$, so the
bracket has norm at most $M(1-t)\lVert \mathbf{e}_k\rVert$ and

$$
\lVert H_k\mathbf{e}_k - \mathbf{g}_k\rVert \le \lVert \mathbf{e}_k\rVert^2 M\int_0^1(1-t)\,dt = \frac{M}{2}\lVert \mathbf{e}_k\rVert^2
$$

**Step 4 (control $H_k^{-1}$).** Let $\mu = \lambda_{\min}(\nabla^2 f(\mathbf{x}^*)) \gt 0$. By Lipschitz
continuity and Weyl's inequality, $\lambda_{\min}(H_k) \ge \mu - M\lVert \mathbf{e}_k\rVert \ge \mu/2$
whenever $\lVert \mathbf{e}_k\rVert \le \frac{\mu}{2M}$, so $\lVert H_k^{-1}\rVert \le 2/\mu$ there.

**Step 5 (combine).** On that ball,

$$
\lVert \mathbf{e}_{k+1}\rVert \le \lVert H_k^{-1}\rVert\,\lVert H_k\mathbf{e}_k - \mathbf{g}_k\rVert \le \frac{2}{\mu}\cdot\frac{M}{2}\lVert \mathbf{e}_k\rVert^2 = \frac{M}{\mu}\lVert \mathbf{e}_k\rVert^2
$$

**Step 6 (the region of attraction is invariant).** If additionally
$\lVert \mathbf{e}_0\rVert \le \frac{\mu}{2M}$, then
$\lVert \mathbf{e}_1\rVert \le \frac{M}{\mu}\lVert \mathbf{e}_0\rVert^2 \le \frac{1}{2}\lVert \mathbf{e}_0\rVert$,
so the iterates stay in the ball and the estimate applies at every step; the errors then satisfy
$\frac{M}{\mu}\lVert \mathbf{e}_k\rVert \le 2^{-2^k}$ — the doubling of digits.

$$
\boxed{\lVert \mathbf{x}_{k+1}-\mathbf{x}^*\rVert \le \frac{M}{\mu}\lVert \mathbf{x}_k - \mathbf{x}^*\rVert^2 \quad \text{for } \lVert \mathbf{x}_0 - \mathbf{x}^*\rVert \le \frac{\mu}{2M}}
$$

> **Key takeaway:** The quadratic constant $M/\mu$ is a *ratio of third-order to second-order* information; when the Hessian varies slowly relative to its own smallest eigenvalue, Newton's basin is large and its convergence is explosive.

**Verification.** The cell below recomputes every number claimed in Problem L3.1.

In [18]:
# verify  ||e_{k+1}|| <= (M/mu) ||e_k||^2  on f(u) = sum(exp(u) - u), x* = 0
f31 = lambda u: np.sum(np.exp(u) - u)
g31 = lambda u: np.exp(u) - 1.0
h31 = lambda u: np.diag(np.exp(u))

R = 0.4                                   # work on the ball ||e|| <= R
mu = 1.0                                  # lambda_min(Hessian at x*) = 1
M = np.exp(R)                             # Lipschitz constant of the Hessian on that ball
basin = mu/(2*M)
print(f"mu = {mu},  M = {M:.6f},  guaranteed basin radius mu/(2M) = {basin:.6f}")

x = np.array([0.2, -0.15, 0.1])
assert np.linalg.norm(x) <= basin
errs = [np.linalg.norm(x)]
for _ in range(5):
    x = x - np.linalg.solve(h31(x), g31(x))
    errs.append(np.linalg.norm(x))
errs = np.array(errs)
ok = errs[:-1] > 1e-11                       # ignore the round-off floor
ratios = errs[1:][ok]/errs[:-1][ok]**2
print("errors:", np.array2string(errs, precision=12))
print("observed e_(k+1)/e_k^2:", np.array2string(ratios, precision=6), f"  bound M/mu = {M/mu:.6f}")
print("t_k = (M/mu) e_k  vs the 2^(-2^k) envelope:")
for k, e in enumerate(errs[:4]):
    print(f"  k={k}: t_k = {M/mu*e:.3e}   2^(-2^k) = {2.0**(-2**k):.3e}")
assert np.all(ratios <= M/mu + 1e-12)
assert np.all(M/mu*errs[:4] <= 2.0**(-2**np.arange(4)) + 1e-12)

mu = 1.0,  M = 1.491825,  guaranteed basin radius mu/(2M) = 0.335160
errors: [0.269258240357 0.022677985472 0.000188128959 0.000000015388
 0.             0.            ]
observed e_(k+1)/e_k^2: [0.3128   0.365803 0.434789 0.485453]   bound M/mu = 1.491825
t_k = (M/mu) e_k  vs the 2^(-2^k) envelope:
  k=0: t_k = 4.017e-01   2^(-2^k) = 5.000e-01
  k=1: t_k = 3.383e-02   2^(-2^k) = 2.500e-01
  k=2: t_k = 2.807e-04   2^(-2^k) = 6.250e-02
  k=3: t_k = 2.296e-08   2^(-2^k) = 3.906e-03


### Problem L3.2 — Zoutendijk's Theorem and Global Convergence

**Problem Statement:** Let $f$ be bounded below and $L$-smooth on an open set containing the level set
$\{f \le f(\mathbf{x}_0)\}$, and let $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k\mathbf{d}_k$ with
$\mathbf{d}_k$ a descent direction and $\alpha_k$ satisfying the Wolfe conditions. Prove Zoutendijk's
condition $\sum_k \cos^2\theta_k \lVert \nabla f_k\rVert^2 \lt \infty$, where
$\cos\theta_k = \frac{-\nabla f_k^T\mathbf{d}_k}{\lVert \nabla f_k\rVert\lVert \mathbf{d}_k\rVert}$, and
deduce global convergence when the directions stay bounded away from orthogonality with $-\nabla f$.

*Intuition:* Wolfe (W2) forces steps to be long enough to be worth taking; smoothness caps how long they can be; together they convert total decrease into a convergent series.

**Solution:**

**Step 1 (lower bound on the step from (W2) and smoothness).** Subtracting $\nabla f_k^T\mathbf{d}_k$ from
the curvature condition $\nabla f_{k+1}^T\mathbf{d}_k \ge c_2\nabla f_k^T\mathbf{d}_k$ gives

$$
\left(\nabla f_{k+1} - \nabla f_k\right)^T\mathbf{d}_k \ge (c_2 - 1)\nabla f_k^T\mathbf{d}_k \gt 0
$$

By Cauchy-Schwarz and $L$-smoothness, the left side is at most
$L\lVert \mathbf{x}_{k+1}-\mathbf{x}_k\rVert\lVert \mathbf{d}_k\rVert = L\alpha_k\lVert \mathbf{d}_k\rVert^2$.
Hence

$$
\alpha_k \ge \frac{(c_2-1)\,\nabla f_k^T\mathbf{d}_k}{L\lVert \mathbf{d}_k\rVert^2} = \frac{(1-c_2)\lvert \nabla f_k^T\mathbf{d}_k\rvert}{L\lVert \mathbf{d}_k\rVert^2}
$$

**Step 2 (feed into (W1)).** Armijo says
$f_{k+1} \le f_k + c_1\alpha_k\nabla f_k^T\mathbf{d}_k = f_k - c_1\alpha_k\lvert \nabla f_k^T\mathbf{d}_k\rvert$.
Substituting the lower bound on $\alpha_k$:

$$
f_{k+1} \le f_k - \frac{c_1(1-c_2)}{L}\cdot\frac{\left(\nabla f_k^T\mathbf{d}_k\right)^2}{\lVert \mathbf{d}_k\rVert^2} = f_k - c\,\cos^2\theta_k\,\lVert \nabla f_k\rVert^2
$$

with $c = \frac{c_1(1-c_2)}{L} \gt 0$, using
$\frac{(\nabla f_k^T\mathbf{d}_k)^2}{\lVert \mathbf{d}_k\rVert^2} = \cos^2\theta_k\lVert \nabla f_k\rVert^2$.

**Step 3 (telescope).** Summing from $0$ to $K$:

$$
c\sum_{k=0}^{K}\cos^2\theta_k\lVert \nabla f_k\rVert^2 \le f_0 - f_{K+1} \le f_0 - f^* \lt \infty
$$

Letting $K \to \infty$ gives **Zoutendijk's condition**

$$
\sum_{k=0}^{\infty}\cos^2\theta_k\,\lVert \nabla f_k\rVert^2 \lt \infty
$$

**Step 4 (global convergence).** A convergent series has vanishing terms, so
$\cos^2\theta_k\lVert \nabla f_k\rVert^2 \to 0$. If the method keeps directions uniformly non-orthogonal to
the steepest descent direction, $\cos\theta_k \ge \delta \gt 0$ for all $k$, then

$$
\lVert \nabla f_k\rVert^2 \le \frac{1}{\delta^2}\cos^2\theta_k\lVert \nabla f_k\rVert^2 \to 0 \quad \Longrightarrow \quad \nabla f_k \to \mathbf{0}
$$

Steepest descent has $\cos\theta_k \equiv 1$. Quasi-Newton with $B_k$ having uniformly bounded condition
number $\kappa(B_k) \le \bar{\kappa}$ has $\cos\theta_k \ge 1/\bar{\kappa}$, so BFGS with a Wolfe line
search is globally convergent under that safeguard.

$$
\boxed{\sum_k \cos^2\theta_k\lVert \nabla f_k\rVert^2 \lt \infty; \quad \cos\theta_k \ge \delta \gt 0 \implies \lVert \nabla f_k\rVert \to 0}
$$

> **Key takeaway:** Zoutendijk cleanly separates the two jobs: the line search supplies the summability, and the *direction rule* supplies the angle bound — so any direction generator can be certified globally convergent by checking one angle condition.

**Verification.** The cell below recomputes every number claimed in Problem L3.2.

In [19]:
# Zoutendijk on a non-quadratic with a strong-Wolfe line search
f32 = lambda v: (1 - v[0])**2 + 100*(v[1] - v[0]**2)**2
g32 = lambda v: np.array([-2*(1 - v[0]) - 400*v[0]*(v[1] - v[0]**2), 200*(v[1] - v[0]**2)])
c1, c2 = 1e-4, 0.9

x = np.array([-1.2, 1.0])
series, cosines, g = 0.0, [], g32(x)
for _ in range(4000):
    d = -g
    out = line_search(f32, g32, x, d, g, c1=c1, c2=c2, maxiter=100)
    a = out[0]
    if a is None:
        break
    cos_t = -(g @ d)/(np.linalg.norm(g)*np.linalg.norm(d))
    cosines.append(cos_t)
    series += cos_t**2 * (g @ g)
    x = x + a*d
    g = g32(x)
    if np.linalg.norm(g) < 1e-8:
        break
L_emp = 2600.0     # a crude Lipschitz constant for grad f on the sublevel set of x0
print(f"steepest descent: cos(theta_k) == 1?  {np.allclose(cosines, 1.0)}")
print(f"Zoutendijk series  = {series:.4f}")
print(f"budget L(f0-f*)/(c1(1-c2)) = {L_emp*(f32(np.array([-1.2, 1.0])) - 0.0)/(c1*(1-c2)):.4e}")
g_start = np.linalg.norm(g32(np.array([-1.2, 1.0])))
print(f"iterations = {len(cosines)},  ||grad f|| went {g_start:.3e} -> {np.linalg.norm(g):.3e}")
print("the series is finite, so its terms tend to 0; with cos(theta_k) = 1 that IS grad f -> 0")
assert np.allclose(cosines, 1.0)
assert series < L_emp*(f32(np.array([-1.2, 1.0])))/(c1*(1-c2))
assert np.linalg.norm(g) < g_start/1e4

steepest descent: cos(theta_k) == 1?  True
Zoutendijk series  = 55799.1336
budget L(f0-f*)/(c1(1-c2)) = 6.2920e+09
iterations = 4000,  ||grad f|| went 2.329e+02 -> 5.631e-04
the series is finite, so its terms tend to 0; with cos(theta_k) = 1 that IS grad f -> 0


### Problem L3.3 — BFGS Preserves Positive Definiteness

**Problem Statement:** Prove that if $B_k \succ 0$ and $\mathbf{s}_k^T\mathbf{y}_k \gt 0$, then the BFGS
update $B_{k+1}$ is positive definite. Then show that the Wolfe curvature condition guarantees
$\mathbf{s}_k^T\mathbf{y}_k \gt 0$, and state the Dennis-Moré characterization of superlinear convergence.

*Intuition:* The update subtracts exactly the piece of $B_k$ that a Cauchy-Schwarz inequality can afford to lose, and adds back a genuinely positive rank-one term.

**Solution:**

**Step 1 (the quadratic form).** Drop indices and let $\mathbf{z} \neq \mathbf{0}$:

$$
\mathbf{z}^TB_{+}\mathbf{z} = \mathbf{z}^TB\mathbf{z} - \frac{\left(\mathbf{z}^TB\mathbf{s}\right)^2}{\mathbf{s}^TB\mathbf{s}} + \frac{\left(\mathbf{y}^T\mathbf{z}\right)^2}{\mathbf{y}^T\mathbf{s}}
$$

The last term is $\ge 0$ because $\mathbf{y}^T\mathbf{s} \gt 0$.

**Step 2 (the first two terms are $\ge 0$ by Cauchy-Schwarz).** Since $B \succ 0$, it has a symmetric
square root $B^{1/2}$. Put $\mathbf{u} = B^{1/2}\mathbf{z}$ and $\mathbf{v} = B^{1/2}\mathbf{s}$. Then

$$
\mathbf{z}^TB\mathbf{z} - \frac{(\mathbf{z}^TB\mathbf{s})^2}{\mathbf{s}^TB\mathbf{s}} = \lVert \mathbf{u}\rVert^2 - \frac{\left(\mathbf{u}^T\mathbf{v}\right)^2}{\lVert \mathbf{v}\rVert^2} \ge 0
$$

by Cauchy-Schwarz, with **equality iff $\mathbf{u} \parallel \mathbf{v}$**, i.e. iff
$\mathbf{z} = c\,\mathbf{s}$ for some $c \neq 0$.

**Step 3 (rule out equality).** Suppose $\mathbf{z}^TB_{+}\mathbf{z} = 0$. Then both nonnegative pieces
vanish. Vanishing of the first forces $\mathbf{z} = c\mathbf{s}$ with $c \neq 0$; substituting into the
second gives

$$
\frac{\left(\mathbf{y}^T\mathbf{z}\right)^2}{\mathbf{y}^T\mathbf{s}} = \frac{c^2\left(\mathbf{y}^T\mathbf{s}\right)^2}{\mathbf{y}^T\mathbf{s}} = c^2\,\mathbf{y}^T\mathbf{s} \gt 0
$$

a contradiction. Hence $\mathbf{z}^TB_{+}\mathbf{z} \gt 0$ for all $\mathbf{z} \neq \mathbf{0}$:
$B_{+} \succ 0$.

**Step 4 (Wolfe supplies the curvature condition).** With $\mathbf{s}_k = \alpha_k\mathbf{d}_k$, the
curvature condition $\nabla f_{k+1}^T\mathbf{d}_k \ge c_2\nabla f_k^T\mathbf{d}_k$ gives

$$
\mathbf{y}_k^T\mathbf{s}_k = \alpha_k\left(\nabla f_{k+1} - \nabla f_k\right)^T\mathbf{d}_k \ge \alpha_k(c_2 - 1)\nabla f_k^T\mathbf{d}_k \gt 0
$$

since $c_2 \lt 1$ and $\nabla f_k^T\mathbf{d}_k \lt 0$. (In practice, codes additionally *skip* the update
when $\mathbf{y}^T\mathbf{s}$ is too small relative to $\lVert \mathbf{s}\rVert\lVert \mathbf{y}\rVert$, or
use Powell damping.)

**Step 5 (Dennis-Moré).** For a quasi-Newton method converging to $\mathbf{x}^*$ with
$\nabla^2 f(\mathbf{x}^*) \succ 0$ and unit steps eventually accepted, convergence is superlinear **iff**

$$
\lim_{k\to\infty} \frac{\left\lVert \left(B_k - \nabla^2 f(\mathbf{x}^*)\right)\mathbf{p}_k\right\rVert}{\lVert \mathbf{p}_k\rVert} = 0
$$

i.e. $B_k$ need not converge to the true Hessian — it only has to be asymptotically correct *along the
directions actually taken*. That is why BFGS is superlinear with an approximation that never becomes exact.

$$
\boxed{B_k \succ 0 \ \text{and}\ \mathbf{s}_k^T\mathbf{y}_k \gt 0 \implies B_{k+1} \succ 0; \ \text{Wolfe (W2)} \implies \mathbf{s}_k^T\mathbf{y}_k \gt 0}
$$

> **Key takeaway:** The line search and the update are not independent components — (W2) exists in large part to keep $\mathbf{s}^T\mathbf{y} \gt 0$, which is exactly the hypothesis that keeps BFGS directions descent directions forever.

**Verification.** The cell below recomputes every number claimed in Problem L3.3.

In [20]:
# B_+ > 0 whenever B > 0 and s.y > 0, and the failure when the curvature condition is dropped
def bfgs_direct(B, s, y):
    Bs = B @ s
    return B - np.outer(Bs, Bs)/(s @ Bs) + np.outer(y, y)/(y @ s)

worst_eig = np.inf
for _ in range(300):
    n = int(rng.integers(2, 7))
    Z = rng.standard_normal((n, n)); B = Z @ Z.T + 0.5*n*np.eye(n)
    s = rng.standard_normal(n)
    Y = rng.standard_normal((n, n)); y = (Y @ Y.T + n*np.eye(n)) @ s    # forces s.y > 0
    Bp = bfgs_direct(B, s, y)
    assert s @ y > 0
    worst_eig = min(worst_eig, np.min(np.linalg.eigvalsh(Bp)))
print(f"over 300 draws with s.y > 0: min eigenvalue of B_+ = {worst_eig:.3e} > 0")

B = np.eye(2); s = np.array([1.0, 0.0]); y = np.array([-1.0, 0.0])     # s.y = -1 < 0
Bp_bad = bfgs_direct(B, s, y)
print(f"dropping the hypothesis (s.y = {s @ y}):  B_+ =\n{Bp_bad}")
print("eigenvalues:", np.linalg.eigvalsh(Bp_bad))
print(f"resulting Newton-like direction from g=(1,0): grad.d = "
      f"{np.array([1.0,0.0]) @ (-np.linalg.solve(Bp_bad + 1e-12*np.eye(2), np.array([1.0,0.0]))):+.4f}")
assert worst_eig > 0
assert np.min(np.linalg.eigvalsh(Bp_bad)) < 0

over 300 draws with s.y > 0: min eigenvalue of B_+ = 8.148e-01 > 0
dropping the hypothesis (s.y = -1.0):  B_+ =
[[-1.  0.]
 [ 0.  1.]]
eigenvalues: [-1.  1.]
resulting Newton-like direction from g=(1,0): grad.d = +1.0000


### Problem L3.4 — A Convex, Smooth Function Where Newton Diverges

**Problem Statement:** Apply Newton's method to $f(x) = \sqrt{1+x^2}$, which is strictly convex, $\mathcal{C}^\infty$,
and has the unique global minimizer $x^* = 0$. Derive the iteration in closed form and characterize exactly
the starting points for which it converges. Explain which hypothesis of the quadratic-convergence theorem
fails, and how a line search fixes it.

*Intuition:* Far from the origin this function is nearly linear, so its curvature is nearly zero — and Newton divides by curvature.

**Solution:**

**Step 1 (derivatives).**

$$
f'(x) = \frac{x}{\sqrt{1+x^2}}, \qquad f''(x) = \frac{1}{\left(1+x^2\right)^{3/2}} \gt 0
$$

so $f$ is strictly convex everywhere, with $x^* = 0$, $f'' (0) = 1 \gt 0$: SOSC holds and the minimizer is
perfectly nondegenerate.

**Step 2 (the iteration collapses to a cubic map).**

$$
x_{k+1} = x_k - \frac{f'(x_k)}{f''(x_k)} = x_k - \frac{x_k}{\sqrt{1+x_k^2}}\left(1+x_k^2\right)^{3/2} = x_k - x_k\left(1+x_k^2\right) = -x_k^3
$$

**Step 3 (exact classification).** Since $x_{k+1} = -x_k^3$ and $3^k$ is odd, induction gives the exact closed form

$$
x_k = (-1)^k\, x_0^{3^k}
$$

so in particular $\lvert x_k\rvert = \lvert x_0\rvert^{3^k}$. Therefore:

- $\lvert x_0\rvert \lt 1$: $\lvert x_k\rvert \to 0$ **cubically** — even faster than quadratic.
- $\lvert x_0\rvert = 1$: $x_k$ alternates in $\{1, -1\}$ forever — a **2-cycle**, no convergence.
- $\lvert x_0\rvert \gt 1$: $\lvert x_k\rvert \to \infty$ **cubically fast** — spectacular divergence
  (from $x_0 = 1.1$: $1.1 \to -1.331 \to 2.3579 \to -13.110 \to 2253.2$, i.e. $\lvert x_4\rvert = 1.1^{81}$).

**Step 4 (which hypothesis fails).** Not convexity, not smoothness, not nondegeneracy — the theorem of
Problem L3.1 is *local*, and its region of attraction $\lVert \mathbf{e}_0\rVert \le \mu/(2M)$ is finite.
Here $f''(x) \to 0$ as $\lvert x\rvert \to \infty$, so $f$ is **not** strongly convex globally: the
effective $\mu$ on any region containing large $x$ is essentially $0$, shrinking the guaranteed basin. The
Newton step $-x(1+x^2)$ blows up because it divides the modest gradient (bounded by $1$) by the vanishing
curvature.

**Step 5 (line search fixes it).** With backtracking from $x_0 = 3$: $f(3) = \sqrt{10} \approx 3.1623$,
$f'(3) = 3/\sqrt{10} \approx 0.9487$, Newton direction $p = -30$, so $f'(x_0)p \approx -28.46$.

| $\alpha$ | $x_0 + \alpha p$ | $f$ | Armijo bound $3.1623 - 10^{-4}(28.46)\alpha$ | verdict |
|---|---|---|---|---|
| $1$ | $-27$ | $27.02$ | $3.1595$ | reject |
| $1/2$ | $-12$ | $12.04$ | $3.1609$ | reject |
| $1/4$ | $-4.5$ | $4.610$ | $3.1616$ | reject |
| $1/8$ | $-0.75$ | $1.250$ | $3.1619$ | **accept** |

One safeguarded step moves from $3$ to $-0.75$, inside the basin $\lvert x\rvert \lt 1$, after which pure
Newton converges cubically. Damped Newton is therefore globally convergent here even though pure Newton is
not.

$$
\boxed{x_{k+1} = -x_k^3: \ \text{converges cubically iff } \lvert x_0\rvert \lt 1, \ \text{cycles at } \lvert x_0\rvert = 1, \ \text{diverges for } \lvert x_0\rvert \gt 1}
$$

> **Key takeaway:** Newton's quadratic convergence is a *local* theorem with a finite basin, and no amount of convexity or smoothness makes it global — which is precisely why every production Newton solver is a *damped* Newton solver.

**Verification.** The cell below recomputes every number claimed in Problem L3.4.

In [21]:
# f(x) = sqrt(1+x^2): the Newton map is exactly x -> -x^3
f34   = lambda t: np.sqrt(1 + t*t)
fp34  = lambda t: t/np.sqrt(1 + t*t)
fpp34 = lambda t: (1 + t*t)**-1.5

for t in [0.5, 1.1, 3.0, -2.0]:
    print(f"  x={t:<5}: Newton map {t - fp34(t)/fpp34(t):+.6f}   -x^3 = {-t**3:+.6f}")
    assert abs((t - fp34(t)/fpp34(t)) - (-t**3)) < 1e-12

for x0 in [0.9, 1.0, 1.1]:
    seq = [x0]
    for _ in range(5):
        seq.append(-seq[-1]**3)
    closed = [(-1.0)**k * x0**(3**k) for k in range(6)]
    print(f"x0={x0}: {np.array2string(np.array(seq), precision=4)}   "
          f"closed form matches: {np.allclose(seq, closed)}")
    assert np.allclose(seq, closed)
print(f"|x_4| from x0=1.1 is 1.1^(3^4) = 1.1^81 = {1.1**81:.4f}")

x, c1, rows = 3.0, 1e-4, []                    # backtracking from x0 = 3
d = -fp34(x)/fpp34(x); slope = fp34(x)*d
a = 1.0
while f34(x + a*d) > f34(x) + c1*a*slope:
    rows.append((a, x + a*d, f34(x + a*d), "reject")); a *= 0.5
rows.append((a, x + a*d, f34(x + a*d), "ACCEPT"))
print(f"\nfrom x0 = 3: p = {d}, slope = {slope:.4f}")
for r in rows:
    print(f"  alpha={r[0]:<7.4f} x={r[1]:<9.4f} f={r[2]:<9.4f} {r[3]}")
assert a == 0.125 and np.isclose(x + a*d, -0.75) and np.isclose(f34(x + a*d), 1.25)
assert abs(1.1**81 - 2253.2402) < 1e-3

  x=0.5  : Newton map -0.125000   -x^3 = -0.125000
  x=1.1  : Newton map -1.331000   -x^3 = -1.331000
  x=3.0  : Newton map -27.000000   -x^3 = -27.000000
  x=-2.0 : Newton map +8.000000   -x^3 = +8.000000
x0=0.9: [ 0.9    -0.729   0.3874 -0.0581  0.0002 -0.    ]   closed form matches: True
x0=1.0: [ 1. -1.  1. -1.  1. -1.]   closed form matches: True
x0=1.1: [ 1.1000e+00 -1.3310e+00  2.3579e+00 -1.3110e+01  2.2532e+03 -1.1440e+10]   closed form matches: True
|x_4| from x0=1.1 is 1.1^(3^4) = 1.1^81 = 2253.2402

from x0 = 3: p = -30.0, slope = -28.4605
  alpha=1.0000  x=-27.0000  f=27.0185   reject
  alpha=0.5000  x=-12.0000  f=12.0416   reject
  alpha=0.2500  x=-4.5000   f=4.6098    reject
  alpha=0.1250  x=-0.7500   f=1.2500    ACCEPT
